# 22. Risk Engine v1のエラーとv2の追加価値検証
出典: FX (2).ipynb、セルindex [46, 47, 48, 49]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 46


In [ ]:
# ============================================================
# USD/JPY
# RISK ENGINE v1
# EXPANDING META WALK-FORWARD TEST
#
# 目的
# ------------------------------------------------------------
# 現在完成している
#
#   ML
#   -> Calibration
#   -> Threshold
#   -> Session
#   -> 30min Exit
#   -> Adaptive Position Sizing
#
# を変更せず、
#
#   ・総Exposure上限
#   ・Drawdown時のSize縮小
#   ・Daily Loss Limit
#
# を追加したとき、
#
# リスク調整後OOS成績が改善するかを検証する。
#
#
# 重要
# ------------------------------------------------------------
# Risk Policyも未来情報を使って選ばない。
#
# 例:
#
# 2024 Test
#
# Policy選択:
#   2020-2023のみ使用
#
# Test:
#   2024
#
# ============================================================


from pathlib import Path
from datetime import timedelta, datetime
import heapq

import numpy as np
import pandas as pd


# ============================================================
# 0. CONFIG
# ============================================================

INITIAL_CAPITAL = 1.0

HOLD_MINUTES = 30

BOOTSTRAP_ITERATIONS = 10000

BOOTSTRAP_BLOCK_DAYS = 20

RANDOM_SEED = 42


# ------------------------------------------------------------
# Risk Policy候補
#
# 結果を見た後で数値を微調整しない。
# まずこの固定候補でOOS検証する。
# ------------------------------------------------------------

RISK_POLICIES = {

    # 現在の戦略そのまま
    "NO_RISK": {
        "gross_cap": None,
        "daily_loss_limit": None,
        "dd_rules": [],
    },

    # 同時Exposure最大2.0
    "CAP_2_0": {
        "gross_cap": 2.0,
        "daily_loss_limit": None,
        "dd_rules": [],
    },

    # やや厳しいExposure制限
    "CAP_1_5": {
        "gross_cap": 1.5,
        "daily_loss_limit": None,
        "dd_rules": [],
    },

    # Drawdownが増えたらPosition縮小
    "DD_SOFT": {
        "gross_cap": None,
        "daily_loss_limit": None,

        # DD 0.75% -> 75%
        # DD 1.25% -> 50%
        "dd_rules": [
            (0.0075, 0.75),
            (0.0125, 0.50),
        ],
    },

    # 1日-0.40%で、その日の新規取引停止
    "DAILY_STOP": {
        "gross_cap": None,
        "daily_loss_limit": 0.0040,
        "dd_rules": [],
    },

    # 全部を組み合わせたModerate設定
    "COMBINED": {
        "gross_cap": 2.0,

        "daily_loss_limit": 0.0040,

        "dd_rules": [
            (0.0075, 0.75),
            (0.0125, 0.50),
        ],
    },

}


# ------------------------------------------------------------
# Risk Policy選択Score
#
# 単純利益ではなく、
#
# Growth
# - MaxDD
# - Tail Risk
#
# を使う。
# ------------------------------------------------------------

MAX_DD_PENALTY = 1.00

CVAR_PENALTY = 0.50


# ============================================================
# 1. DATA CHECK
# ============================================================

if "all_test_trades" not in globals():

    raise RuntimeError(
        "all_test_trades がありません。\n"
        "Position Sizingのコードを先に実行してください。"
    )


trades = all_test_trades.copy()


required_columns = [

    "test_year",
    "position_size",
    "sized_return",

]


missing = [

    col

    for col in required_columns

    if col not in trades.columns

]


if missing:

    raise RuntimeError(

        "all_test_trades に必要な列がありません。\n"
        f"不足列: {missing}"

    )


# ============================================================
# 2. TIMESTAMP
# ============================================================

# indexがDatetimeIndexならそのまま使用
if isinstance(
    trades.index,
    pd.DatetimeIndex
):

    trades["entry_time"] = trades.index


# timestamp列がある場合
elif "timestamp" in trades.columns:

    trades["entry_time"] = pd.to_datetime(
        trades["timestamp"],
        errors="coerce"
    )


else:

    # indexをtimestampとして試す
    trades["entry_time"] = pd.to_datetime(
        trades.index,
        errors="coerce"
    )


if trades["entry_time"].isna().any():

    raise RuntimeError(
        "timestampを正しく取得できない行があります。"
    )


trades = trades.sort_values(
    "entry_time"
).reset_index(
    drop=True
)


trades["_trade_id"] = np.arange(
    len(trades)
)


# ============================================================
# 3. NUMERIC CLEANING
# ============================================================

for col in [

    "position_size",
    "sized_return",

]:

    trades[col] = pd.to_numeric(
        trades[col],
        errors="coerce"
    )


trades["test_year"] = pd.to_numeric(
    trades["test_year"],
    errors="coerce"
)


trades = trades.dropna(
    subset=[
        "entry_time",
        "test_year",
        "position_size",
        "sized_return",
    ]
)


trades["test_year"] = (
    trades["test_year"]
    .astype(int)
)


# ============================================================
# 4. UNIT RETURN
# ============================================================

# Adaptive Sizing後Return:
#
# sized_return
#
# を基準にする。
#
#
# unit_return =
#
# sized_return / position_size
#
#
# これでRisk Engine側がSizeを変えた場合も
# 同じ元戦略Returnを利用できる。
# ============================================================

EPS = 1e-12


trades["unit_return"] = np.where(

    trades["position_size"].abs() > EPS,

    trades["sized_return"]
    /
    trades["position_size"],

    0.0,

)


trades["exit_time"] = (

    trades["entry_time"]

    +

    pd.Timedelta(
        minutes=HOLD_MINUTES
    )

)


# ============================================================
# 5. SANITY CHECK
# ============================================================

reconstructed = (

    trades["unit_return"]

    *

    trades["position_size"]

)


reconstruction_mae = float(

    np.mean(

        np.abs(

            reconstructed
            -
            trades["sized_return"]

        )

    )

)


print()
print("=" * 90)

print(
    "DATA CHECK"
)

print("=" * 90)

print(
    "Trades:",
    len(trades)
)

print(
    "Years:",
    sorted(
        trades["test_year"].unique()
    )
)

print(
    "Return reconstruction MAE:",
    reconstruction_mae
)


# ============================================================
# 6. RISK POLICY
# ============================================================

def drawdown_scale(
    current_drawdown,
    dd_rules
):

    """
    現在のDrawdownに応じたPosition Size倍率。

    current_drawdown:
        -0.01 = -1%

    dd_rules:
        [(0.0075, 0.75), ...]
    """

    dd_magnitude = abs(
        min(
            current_drawdown,
            0.0
        )
    )


    scale = 1.0


    for threshold, rule_scale in sorted(
        dd_rules
    ):

        if dd_magnitude >= threshold:

            scale = rule_scale


    return scale


# ============================================================
# 7. PORTFOLIO SIMULATOR
# ============================================================

def simulate_policy(
    data,
    policy_by_year,
    starting_capital=1.0,
):

    """
    実際の時間順にPositionを処理する。

    重要:
    30分保有なので、
    15分後に次のSignalが来れば
    Positionは重複する。

    gross_capはその重複Exposureを制限する。
    """

    data = (

        data
        .sort_values("entry_time")
        .copy()

    )


    equity = float(
        starting_capital
    )


    high_water = equity


    # --------------------------------------------------------
    # Open Position
    #
    # Heap:
    #
    # (
    #   exit_time,
    #   trade_id,
    #   effective_size,
    #   pnl_amount,
    #   entry_equity
    # )
    # --------------------------------------------------------

    open_positions = []


    gross_open = 0.0


    current_day = None

    day_start_equity = equity


    records = []

    equity_records = []


    max_gross_seen = 0.0


    cap_clipped_count = 0

    daily_stopped_count = 0

    dd_reduced_count = 0

    zero_size_count = 0


    # ========================================================
    # Helper: Position Exit
    # ========================================================

    def process_exits_until(
        current_time
    ):

        nonlocal equity
        nonlocal high_water
        nonlocal gross_open


        while (

            open_positions

            and

            open_positions[0][0]
            <=
            current_time

        ):


            (
                exit_time,
                trade_id,
                effective_size,
                pnl_amount,
                entry_equity,

            ) = heapq.heappop(
                open_positions
            )


            equity += pnl_amount


            gross_open -= effective_size


            gross_open = max(
                0.0,
                gross_open
            )


            high_water = max(
                high_water,
                equity
            )


            equity_records.append({

                "time":
                    exit_time,

                "equity":
                    equity,

            })


    # ========================================================
    # Entry loop
    # ========================================================

    for row in data.itertuples(
        index=False
    ):


        entry_time = row.entry_time

        exit_time = row.exit_time

        year = int(
            row.test_year
        )


        # ----------------------------------------------------
        # まず既にExitしたPositionを精算
        # ----------------------------------------------------

        process_exits_until(
            entry_time
        )


        # ----------------------------------------------------
        # 日付変更
        # ----------------------------------------------------

        day = entry_time.date()


        if current_day != day:

            current_day = day

            day_start_equity = equity


        # ----------------------------------------------------
        # Policy
        # ----------------------------------------------------

        policy_name = policy_by_year.get(
            year,
            "NO_RISK"
        )


        policy = RISK_POLICIES[
            policy_name
        ]


        desired_size = float(
            row.position_size
        )


        original_size = desired_size


        # ----------------------------------------------------
        # Drawdown
        # ----------------------------------------------------

        if high_water > 0:

            current_dd = (

                equity
                /
                high_water

                -
                1.0

            )

        else:

            current_dd = 0.0


        dd_scale = drawdown_scale(

            current_dd,

            policy[
                "dd_rules"
            ],

        )


        if dd_scale < 1.0:

            dd_reduced_count += 1


        desired_size *= dd_scale


        # ----------------------------------------------------
        # Daily Loss Limit
        # ----------------------------------------------------

        daily_loss_limit = policy[
            "daily_loss_limit"
        ]


        daily_block = False


        if (
            daily_loss_limit is not None

            and

            day_start_equity > 0
        ):


            daily_return = (

                equity
                /
                day_start_equity

                -
                1.0

            )


            if daily_return <= -daily_loss_limit:

                desired_size = 0.0

                daily_block = True

                daily_stopped_count += 1


        # ----------------------------------------------------
        # Gross Exposure Cap
        # ----------------------------------------------------

        gross_cap = policy[
            "gross_cap"
        ]


        cap_clipped = False


        if gross_cap is not None:


            available = max(

                0.0,

                gross_cap
                -
                gross_open,

            )


            if desired_size > available:

                desired_size = available

                cap_clipped = True

                cap_clipped_count += 1


        effective_size = max(
            0.0,
            desired_size
        )


        if effective_size <= EPS:

            zero_size_count += 1


        # ----------------------------------------------------
        # Entry Equity
        # ----------------------------------------------------

        entry_equity = equity


        # ----------------------------------------------------
        # Future PnL
        #
        # 決定時にはfuture returnは使っていない。
        #
        # Exit PnL計算のためだけに使用する。
        # ----------------------------------------------------

        pnl_amount = (

            entry_equity

            *

            effective_size

            *

            float(
                row.unit_return
            )

        )


        # ----------------------------------------------------
        # Position登録
        # ----------------------------------------------------

        if effective_size > EPS:


            heapq.heappush(

                open_positions,

                (

                    exit_time,

                    int(
                        row._trade_id
                    ),

                    effective_size,

                    pnl_amount,

                    entry_equity,

                ),

            )


            gross_open += effective_size


            max_gross_seen = max(

                max_gross_seen,

                gross_open,

            )


        # ----------------------------------------------------
        # Trade Record
        # ----------------------------------------------------

        records.append({

            "trade_id":
                int(
                    row._trade_id
                ),

            "entry_time":
                entry_time,

            "exit_time":
                exit_time,

            "test_year":
                year,

            "policy":
                policy_name,

            "original_size":
                original_size,

            "effective_size":
                effective_size,

            "dd_scale":
                dd_scale,

            "current_dd":
                current_dd,

            "gross_before":
                gross_open
                -
                effective_size,

            "gross_after":
                gross_open,

            "cap_clipped":
                cap_clipped,

            "daily_block":
                daily_block,

            "unit_return":
                float(
                    row.unit_return
                ),

            "trade_return":
                (
                    effective_size
                    *
                    float(
                        row.unit_return
                    )
                ),

            "pnl_amount":
                pnl_amount,

        })


    # ========================================================
    # 最後のPositionを全部Exit
    # ========================================================

    process_exits_until(
        pd.Timestamp.max.tz_localize(
            None
        )
        if data["exit_time"].dt.tz is None
        else
        data["exit_time"].max()
        +
        pd.Timedelta(
            days=1
        )
    )


    trade_records = pd.DataFrame(
        records
    )


    equity_df = pd.DataFrame(
        equity_records
    )


    if not equity_df.empty:

        equity_df = (

            equity_df

            .sort_values(
                "time"
            )

            .drop_duplicates(
                subset="time",
                keep="last"
            )

            .set_index(
                "time"
            )

        )


    summary = {

        "starting_capital":
            starting_capital,

        "ending_capital":
            equity,

        "growth":
            (
                equity
                /
                starting_capital

                -
                1.0
            ),

        "max_gross_exposure":
            max_gross_seen,

        "cap_clipped_count":
            cap_clipped_count,

        "daily_stopped_count":
            daily_stopped_count,

        "dd_reduced_count":
            dd_reduced_count,

        "zero_size_count":
            zero_size_count,

    }


    return (
        trade_records,
        equity_df,
        summary
    )


# ============================================================
# 8. EQUITY -> DAILY RETURN
# ============================================================

def make_daily_equity(
    equity_df,
    starting_capital=1.0
):

    if equity_df.empty:

        return pd.Series(
            dtype=float
        )


    eq = equity_df[
        "equity"
    ].copy()


    start_day = (
        eq.index.min()
        .normalize()
    )


    end_day = (
        eq.index.max()
        .normalize()
    )


    daily_index = pd.date_range(

        start=start_day,

        end=end_day,

        freq="1D",

        tz=eq.index.tz,

    )


    daily_equity = (

        eq

        .resample(
            "1D"
        )

        .last()

        .reindex(
            daily_index
        )

        .ffill()

    )


    daily_equity.iloc[0] = (

        daily_equity.iloc[0]

        if np.isfinite(
            daily_equity.iloc[0]
        )

        else starting_capital

    )


    daily_returns = (

        daily_equity

        .pct_change()

        .fillna(
            0.0
        )

    )


    return pd.DataFrame({

        "equity":
            daily_equity,

        "daily_return":
            daily_returns,

    })


# ============================================================
# 9. PERFORMANCE STATS
# ============================================================

def calc_performance(
    trade_records,
    equity_df,
    starting_capital=1.0
):


    if trade_records.empty:

        return {}


    r = (

        trade_records[
            "trade_return"
        ]

        .astype(float)

    )


    positive = r[
        r > 0
    ].sum()


    negative = -r[
        r < 0
    ].sum()


    if negative > 0:

        pf = positive / negative

    elif positive > 0:

        pf = np.inf

    else:

        pf = np.nan


    daily = make_daily_equity(

        equity_df,

        starting_capital,

    )


    if daily.empty:

        return {}


    equity = daily[
        "equity"
    ]


    peak = equity.cummax()


    dd = (

        equity
        /
        peak

        -
        1.0

    )


    max_dd = float(
        dd.min()
    )


    growth = float(

        equity.iloc[-1]

        /
        starting_capital

        -
        1.0

    )


    daily_returns = daily[
        "daily_return"
    ]


    worst_day = float(
        daily_returns.min()
    )


    # --------------------------------------------------------
    # Historical CVaR 95%
    #
    # 最悪5%の日の平均損失
    # 正の数字として表示
    # --------------------------------------------------------

    q05 = daily_returns.quantile(
        0.05
    )


    worst_tail = daily_returns[
        daily_returns <= q05
    ]


    if len(
        worst_tail
    ) > 0:

        cvar_95 = float(
            -worst_tail.mean()
        )

    else:

        cvar_95 = np.nan


    if max_dd < 0:

        return_to_dd = (

            growth
            /
            abs(
                max_dd
            )

        )

    else:

        return_to_dd = np.nan


    if daily_returns.std(
        ddof=1
    ) > 0:

        daily_sharpe = (

            daily_returns.mean()

            /
            daily_returns.std(
                ddof=1
            )

            *

            np.sqrt(
                252
            )

        )

    else:

        daily_sharpe = np.nan


    return {

        "trades":
            len(
                trade_records
            ),

        "active_trades":
            int(
                (
                    trade_records[
                        "effective_size"
                    ]
                    >
                    EPS
                ).sum()
            ),

        "mean_effective_size":
            float(
                trade_records[
                    "effective_size"
                ].mean()
            ),

        "win_rate":
            float(
                (
                    r > 0
                ).mean()
            ),

        "avg_trade_return":
            float(
                r.mean()
            ),

        "profit_factor":
            float(
                pf
            ),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            float(
                return_to_dd
            ),

        "daily_sharpe":
            float(
                daily_sharpe
            ),

        "worst_day":
            worst_day,

        "daily_cvar_95":
            cvar_95,

    }


# ============================================================
# 10. POLICY SCORE
# ============================================================

def policy_score(
    stats
):

    if not stats:

        return -np.inf


    growth = stats[
        "growth"
    ]


    max_dd = abs(
        stats[
            "max_dd"
        ]
    )


    cvar = stats[
        "daily_cvar_95"
    ]


    pf = stats[
        "profit_factor"
    ]


    if not np.isfinite(
        growth
    ):

        return -np.inf


    if not np.isfinite(
        pf
    ):

        return -np.inf


    # Edge自体が消えているPolicyは採用しない
    if pf <= 1.0:

        return -np.inf


    score = (

        growth

        -

        MAX_DD_PENALTY
        *
        max_dd

        -

        CVAR_PENALTY
        *
        cvar

    )


    return float(
        score
    )


# ============================================================
# 11. STATIC POLICY TEST
#
# 診断用。
#
# 全期間から一番を選んではいけない。
# ============================================================

print()
print("=" * 90)

print(
    "STATIC POLICY COMPARISON"
)

print(
    "(DIAGNOSTIC ONLY - NOT USED FOR FINAL SELECTION)"
)

print("=" * 90)


static_rows = []


for policy_name in RISK_POLICIES:


    mapping = {

        int(year):
            policy_name

        for year in trades[
            "test_year"
        ].unique()

    }


    rec, eq, sim_summary = simulate_policy(

        trades,

        mapping,

        starting_capital=
            INITIAL_CAPITAL,

    )


    stats = calc_performance(

        rec,

        eq,

        INITIAL_CAPITAL,

    )


    static_rows.append({

        "policy":
            policy_name,

        **stats,

        **sim_summary,

        "score":
            policy_score(
                stats
            ),

    })


static_results = pd.DataFrame(
    static_rows
)


static_show = static_results.copy()


for col in [

    "win_rate",
    "avg_trade_return",
    "growth",
    "max_dd",
    "worst_day",
    "daily_cvar_95",

]:

    if col in static_show.columns:

        static_show[
            col
        ] *= 100


print(

    static_show[
        [

            "policy",
            "growth",
            "max_dd",
            "profit_factor",
            "return_to_dd",
            "daily_sharpe",
            "worst_day",
            "daily_cvar_95",
            "max_gross_exposure",
            "cap_clipped_count",
            "daily_stopped_count",
            "dd_reduced_count",
            "score",

        ]
    ]

    .sort_values(
        "score",
        ascending=False
    )

    .to_string(
        index=False
    )

)


# ============================================================
# 12. EXPANDING WALK-FORWARD POLICY SELECTION
# ============================================================

years = sorted(
    trades[
        "test_year"
    ].unique()
)


if len(
    years
) < 2:

    raise RuntimeError(
        "Risk Walk-Forwardには最低2年必要です。"
    )


warmup_year = years[0]


evaluation_years = years[
    1:
]


print()
print("=" * 90)

print(
    "EXPANDING META WALK-FORWARD POLICY SELECTION"
)

print("=" * 90)


print(
    "Warm-up year:",
    warmup_year
)

print(
    "Evaluation years:",
    evaluation_years
)


selection_rows = []

selected_policy_by_year = {}


# Warm-up yearはRiskなし
selected_policy_by_year[
    warmup_year
] = "NO_RISK"


for test_year in evaluation_years:


    historical = trades.loc[

        trades[
            "test_year"
        ]
        <
        test_year

    ].copy()


    candidate_rows = []


    for policy_name in RISK_POLICIES:


        mapping = {

            int(year):
                policy_name

            for year in historical[
                "test_year"
            ].unique()

        }


        rec, eq, sim_summary = simulate_policy(

            historical,

            mapping,

            starting_capital=
                INITIAL_CAPITAL,

        )


        stats = calc_performance(

            rec,

            eq,

            INITIAL_CAPITAL,

        )


        score = policy_score(
            stats
        )


        candidate_rows.append({

            "test_year":
                int(
                    test_year
                ),

            "policy":
                policy_name,

            "history_start":
                int(
                    historical[
                        "test_year"
                    ].min()
                ),

            "history_end":
                int(
                    historical[
                        "test_year"
                    ].max()
                ),

            "history_trades":
                len(
                    historical
                ),

            "score":
                score,

            "growth":
                stats[
                    "growth"
                ],

            "max_dd":
                stats[
                    "max_dd"
                ],

            "pf":
                stats[
                    "profit_factor"
                ],

            "return_to_dd":
                stats[
                    "return_to_dd"
                ],

            "cvar_95":
                stats[
                    "daily_cvar_95"
                ],

        })


    candidate_table = pd.DataFrame(
        candidate_rows
    )


    candidate_table = candidate_table.sort_values(

        [
            "score",
            "pf",
        ],

        ascending=[
            False,
            False,
        ],

    )


    winner = candidate_table.iloc[
        0
    ]


    chosen_policy = winner[
        "policy"
    ]


    selected_policy_by_year[
        int(
            test_year
        )
    ] = chosen_policy


    selection_rows.extend(
        candidate_rows
    )


    print()
    print("-" * 70)

    print(
        "TEST YEAR:",
        test_year
    )

    print(
        "History:",
        int(
            historical[
                "test_year"
            ].min()
        ),
        "->",
        int(
            historical[
                "test_year"
            ].max()
        )
    )

    print(
        "Selected Risk Policy:",
        chosen_policy
    )

    print(
        candidate_table[
            [
                "policy",
                "score",
                "growth",
                "max_dd",
                "pf",
                "return_to_dd",
                "cvar_95",
            ]
        ]
        .head(
            len(
                RISK_POLICIES
            )
        )
        .to_string(
            index=False
        )
    )


selection_results = pd.DataFrame(
    selection_rows
)


# ============================================================
# 13. FINAL META-OOS DATA
# ============================================================

# Warm-up年は最終評価から除く
meta_data = trades.loc[

    trades[
        "test_year"
    ].isin(
        evaluation_years
    )

].copy()


# ============================================================
# 14. BASELINE
#
# Current Adaptive Position Sizing
# ============================================================

baseline_mapping = {

    int(year):
        "NO_RISK"

    for year in evaluation_years

}


baseline_records, baseline_eq, baseline_summary = (

    simulate_policy(

        meta_data,

        baseline_mapping,

        starting_capital=
            INITIAL_CAPITAL,

    )

)


baseline_stats = calc_performance(

    baseline_records,

    baseline_eq,

    INITIAL_CAPITAL,

)


# ============================================================
# 15. META WALK-FORWARD RISK ENGINE
# ============================================================

risk_mapping = {

    int(year):
        selected_policy_by_year[
            int(year)
        ]

    for year in evaluation_years

}


risk_records, risk_eq, risk_summary = simulate_policy(

    meta_data,

    risk_mapping,

    starting_capital=
        INITIAL_CAPITAL,

)


risk_stats = calc_performance(

    risk_records,

    risk_eq,

    INITIAL_CAPITAL,

)


# ============================================================
# 16. OVERALL RESULT
# ============================================================

print()
print("=" * 90)

print(
    "META WALK-FORWARD OOS RESULT"
)

print("=" * 90)


overall = pd.DataFrame({

    "BASELINE":
        {
            **baseline_stats,
            **baseline_summary,
        },

    "RISK_ENGINE":
        {
            **risk_stats,
            **risk_summary,
        },

}).T


overall_show = overall.copy()


for col in [

    "win_rate",
    "avg_trade_return",
    "growth",
    "max_dd",
    "worst_day",
    "daily_cvar_95",

]:

    if col in overall_show.columns:

        overall_show[
            col
        ] *= 100


print(
    overall_show.to_string()
)


# ============================================================
# 17. YEARLY META-OOS RESULT
# ============================================================

print()
print("=" * 90)

print(
    "YEARLY META-OOS RESULT"
)

print("=" * 90)


annual_rows = []


for year in evaluation_years:


    year_data = trades.loc[

        trades[
            "test_year"
        ]
        ==
        year

    ].copy()


    # Baseline
    b_map = {
        int(year):
            "NO_RISK"
    }


    b_rec, b_eq, b_sum = simulate_policy(

        year_data,

        b_map,

        INITIAL_CAPITAL,

    )


    b_stats = calc_performance(

        b_rec,

        b_eq,

        INITIAL_CAPITAL,

    )


    # Risk
    policy_name = selected_policy_by_year[
        int(year)
    ]


    r_map = {
        int(year):
            policy_name
    }


    r_rec, r_eq, r_sum = simulate_policy(

        year_data,

        r_map,

        INITIAL_CAPITAL,

    )


    r_stats = calc_performance(

        r_rec,

        r_eq,

        INITIAL_CAPITAL,

    )


    annual_rows.append({

        "test_year":
            int(year),

        "selected_policy":
            policy_name,

        "base_growth":
            b_stats[
                "growth"
            ],

        "risk_growth":
            r_stats[
                "growth"
            ],

        "base_pf":
            b_stats[
                "profit_factor"
            ],

        "risk_pf":
            r_stats[
                "profit_factor"
            ],

        "base_max_dd":
            b_stats[
                "max_dd"
            ],

        "risk_max_dd":
            r_stats[
                "max_dd"
            ],

        "base_return_dd":
            b_stats[
                "return_to_dd"
            ],

        "risk_return_dd":
            r_stats[
                "return_to_dd"
            ],

        "base_cvar":
            b_stats[
                "daily_cvar_95"
            ],

        "risk_cvar":
            r_stats[
                "daily_cvar_95"
            ],

        "cap_clipped":
            r_sum[
                "cap_clipped_count"
            ],

        "daily_stopped":
            r_sum[
                "daily_stopped_count"
            ],

        "dd_reduced":
            r_sum[
                "dd_reduced_count"
            ],

    })


annual_results = pd.DataFrame(
    annual_rows
)


annual_show = annual_results.copy()


for col in [

    "base_growth",
    "risk_growth",
    "base_max_dd",
    "risk_max_dd",
    "base_cvar",
    "risk_cvar",

]:

    annual_show[
        col
    ] *= 100


print(
    annual_show.to_string(
        index=False
    )
)


# ============================================================
# 18. POLICY SELECTION FREQUENCY
# ============================================================

print()
print("=" * 90)

print(
    "RISK POLICY SELECTION FREQUENCY"
)

print("=" * 90)


selection_frequency = pd.Series(
    risk_mapping
).value_counts()


print(
    selection_frequency
)


# ============================================================
# 19. DAILY RETURN ALIGNMENT
# ============================================================

baseline_daily = make_daily_equity(

    baseline_eq,

    INITIAL_CAPITAL,

)


risk_daily = make_daily_equity(

    risk_eq,

    INITIAL_CAPITAL,

)


daily_compare = pd.concat(

    [

        baseline_daily[
            "daily_return"
        ].rename(
            "baseline_return"
        ),

        risk_daily[
            "daily_return"
        ].rename(
            "risk_return"
        ),

    ],

    axis=1,

).fillna(
    0.0
)


daily_compare[
    "delta"
] = (

    daily_compare[
        "risk_return"
    ]

    -

    daily_compare[
        "baseline_return"
    ]

)


# ============================================================
# 20. MOVING BLOCK BOOTSTRAP
# ============================================================

def moving_block_bootstrap(
    values,
    block_size=20,
    iterations=10000,
    seed=42,
):


    x = np.asarray(
        values,
        dtype=float
    )


    x = x[
        np.isfinite(
            x
        )
    ]


    n = len(
        x
    )


    if n < block_size:

        return {

            "observed":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "prob_positive":
                np.nan,

        }


    rng = np.random.default_rng(
        seed
    )


    samples = []


    blocks_needed = int(

        np.ceil(
            n
            /
            block_size
        )

    )


    max_start = (

        n
        -
        block_size

    )


    for _ in range(
        iterations
    ):


        pieces = []


        for __ in range(
            blocks_needed
        ):


            start = int(

                rng.integers(

                    0,

                    max_start
                    +
                    1

                )

            )


            pieces.append(

                x[
                    start
                    :
                    start
                    +
                    block_size
                ]

            )


        sample = np.concatenate(
            pieces
        )[:n]


        samples.append(
            np.mean(
                sample
            )
        )


    samples = np.asarray(
        samples
    )


    return {

        "observed":
            float(
                np.mean(x)
            ),

        "ci_low":
            float(
                np.percentile(
                    samples,
                    2.5
                )
            ),

        "ci_high":
            float(
                np.percentile(
                    samples,
                    97.5
                )
            ),

        "prob_positive":
            float(
                np.mean(
                    samples > 0
                )
            ),

    }


bootstrap = moving_block_bootstrap(

    daily_compare[
        "delta"
    ],

    block_size=
        BOOTSTRAP_BLOCK_DAYS,

    iterations=
        BOOTSTRAP_ITERATIONS,

    seed=
        RANDOM_SEED,

)


print()
print("=" * 90)

print(
    "RISK ENGINE DAILY RETURN BOOTSTRAP"
)

print("=" * 90)


print(
    "Observed Mean Delta:",
    bootstrap[
        "observed"
    ]
    *
    100,
    "%"
)


print(
    "95% CI:",
    bootstrap[
        "ci_low"
    ]
    *
    100,
    "%",
    "~",
    bootstrap[
        "ci_high"
    ]
    *
    100,
    "%"
)


print(
    "P(Risk Engine Return > Baseline):",
    bootstrap[
        "prob_positive"
    ]
    *
    100,
    "%"
)


# ============================================================
# 21. RISK VALUE DIAGNOSIS
# ============================================================

base_growth = baseline_stats[
    "growth"
]


risk_growth = risk_stats[
    "growth"
]


base_dd = abs(
    baseline_stats[
        "max_dd"
    ]
)


risk_dd = abs(
    risk_stats[
        "max_dd"
    ]
)


base_cvar = baseline_stats[
    "daily_cvar_95"
]


risk_cvar = risk_stats[
    "daily_cvar_95"
]


base_rdd = baseline_stats[
    "return_to_dd"
]


risk_rdd = risk_stats[
    "return_to_dd"
]


# ------------------------------------------------------------
# Growth Retention
# ------------------------------------------------------------

if base_growth > 0:

    growth_retention = (

        risk_growth
        /
        base_growth

    )

else:

    growth_retention = np.nan


# ------------------------------------------------------------
# DD Improvement
# ------------------------------------------------------------

if base_dd > 0:

    dd_improvement = (

        base_dd
        -
        risk_dd

    ) / base_dd

else:

    dd_improvement = np.nan


# ------------------------------------------------------------
# CVaR Improvement
# ------------------------------------------------------------

if base_cvar > 0:

    cvar_improvement = (

        base_cvar
        -
        risk_cvar

    ) / base_cvar

else:

    cvar_improvement = np.nan


# ------------------------------------------------------------
# Return/DD Improvement
# ------------------------------------------------------------

if base_rdd > 0:

    return_dd_improvement = (

        risk_rdd
        /
        base_rdd

        -
        1.0

    )

else:

    return_dd_improvement = np.nan


print()
print("=" * 90)

print(
    "AUTOMATIC RISK DIAGNOSIS"
)

print("=" * 90)


print(
    "Growth Retention:",
    growth_retention
    *
    100,
    "%"
)


print(
    "Max DD Improvement:",
    dd_improvement
    *
    100,
    "%"
)


print(
    "Daily CVaR Improvement:",
    cvar_improvement
    *
    100,
    "%"
)


print(
    "Return/DD Improvement:",
    return_dd_improvement
    *
    100,
    "%"
)


# ============================================================
# 22. YEAR STABILITY
# ============================================================

annual_results[
    "dd_better"
] = (

    annual_results[
        "risk_max_dd"
    ].abs()

    <

    annual_results[
        "base_max_dd"
    ].abs()

)


annual_results[
    "return_dd_better"
] = (

    annual_results[
        "risk_return_dd"
    ]

    >

    annual_results[
        "base_return_dd"
    ]

)


annual_results[
    "growth_positive"
] = (

    annual_results[
        "risk_growth"
    ]

    >
    0

)


print()
print(
    "DD better years:",
    int(
        annual_results[
            "dd_better"
        ].sum()
    ),
    "/",
    len(
        annual_results
    )
)


print(
    "Return/DD better years:",
    int(
        annual_results[
            "return_dd_better"
        ].sum()
    ),
    "/",
    len(
        annual_results
    )
)


print(
    "Positive Growth years:",
    int(
        annual_results[
            "growth_positive"
        ].sum()
    ),
    "/",
    len(
        annual_results
    )
)


# ============================================================
# 23. FINAL DECISION
# ============================================================

print()
print("=" * 90)

print(
    "FINAL DECISION"
)

print("=" * 90)


clear_risk_value = (

    np.isfinite(
        growth_retention
    )

    and

    np.isfinite(
        dd_improvement
    )

    and

    np.isfinite(
        return_dd_improvement
    )

    and

    growth_retention >= 0.90

    and

    dd_improvement >= 0.10

    and

    return_dd_improvement >= 0.10

)


risk_value_with_cost = (

    np.isfinite(
        growth_retention
    )

    and

    np.isfinite(
        dd_improvement
    )

    and

    growth_retention >= 0.80

    and

    dd_improvement >= 0.15

)


if clear_risk_value:


    print(
        "RESULT: RISK ENGINE HAS CLEAR OOS VALUE"
    )

    print()

    print(
        "利益の90%以上を維持しつつ、"
    )

    print(
        "Drawdown / Return-to-DDを改善しています。"
    )

    print()

    print(
        "NEXT:"
    )

    print(
        "Risk Engineを候補として固定し、"
    )

    print(
        "Machine Learning Model Tournamentへ進みます。"
    )


elif risk_value_with_cost:


    print(
        "RESULT: RISK ENGINE REDUCES RISK WITH RETURN COST"
    )

    print()

    print(
        "リターンを多少犠牲にしてRiskを削減しています。"
    )

    print()

    print(
        "採用するかはPaper Trading前に再判断します。"
    )


else:


    print(
        "RESULT: NO CLEAR OOS RISK ENGINE VALUE"
    )

    print()

    print(
        "現在のAdaptive Position Sizingを維持します。"
    )

    print()

    print(
        "Risk Policyのパラメータを結果に合わせて"
    )

    print(
        "再最適化することはしません。"
    )

    print()

    print(
        "NEXT:"
    )

    print(
        "Machine Learning Model Tournamentへ進みます。"
    )


# ============================================================
# 24. SAVE
# ============================================================

OUTPUT_DIR = (

    Path.cwd()

    /

    (
        "risk_engine_v1_"
        +
        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )
    )

)


OUTPUT_DIR.mkdir(
    exist_ok=False
)


static_results.to_csv(

    OUTPUT_DIR
    /
    "static_policy_comparison.csv",

    index=False,

)


selection_results.to_csv(

    OUTPUT_DIR
    /
    "walk_forward_policy_selection.csv",

    index=False,

)


annual_results.to_csv(

    OUTPUT_DIR
    /
    "annual_meta_oos_results.csv",

    index=False,

)


baseline_records.to_csv(

    OUTPUT_DIR
    /
    "baseline_trade_records.csv",

    index=False,

)


risk_records.to_csv(

    OUTPUT_DIR
    /
    "risk_engine_trade_records.csv",

    index=False,

)


daily_compare.to_csv(

    OUTPUT_DIR
    /
    "daily_return_comparison.csv",

)


overall.to_csv(

    OUTPUT_DIR
    /
    "overall_result.csv",

)


print()
print("=" * 90)

print(
    "FINISHED"
)

print("=" * 90)


print(
    OUTPUT_DIR.resolve()
)


print()
print(
    "スクショしてほしい場所:"
)

print(
    "1. STATIC POLICY COMPARISON"
)

print(
    "2. EXPANDING META WALK-FORWARD POLICY SELECTION"
)

print(
    "3. META WALK-FORWARD OOS RESULT"
)

print(
    "4. YEARLY META-OOS RESULT"
)

print(
    "5. RISK POLICY SELECTION FREQUENCY"
)

print(
    "6. RISK ENGINE DAILY RETURN BOOTSTRAP"
)

print(
    "7. AUTOMATIC RISK DIAGNOSIS"
)

print(
    "8. FINAL DECISION"
)


## 元セルindex 47


In [ ]:
# ============================================================
# RISK ENGINE v1 - HOTFIX
#
# 修正内容
# 1. _trade_id -> trade_id
# 2. pandas itertuples の列名変換問題を回避
# 3. timezoneを安全に処理
# 4. Open Position処理を安定化
# ============================================================

import heapq
import numpy as np
import pandas as pd


# ============================================================
# 1. trade_id を安全な列名で作り直す
# ============================================================

if "trades" not in globals():
    raise RuntimeError(
        "trades がありません。\n"
        "前回のRisk EngineコードのDATA CHECK部分まで先に実行してください。"
    )


trades = trades.copy()


# 古い問題のある列を削除
if "_trade_id" in trades.columns:
    trades = trades.drop(columns=["_trade_id"])


# 普通の列名でID作成
trades["trade_id"] = np.arange(
    len(trades),
    dtype=int
)


print("trade_id 修正完了")
print("Trades:", len(trades))
print("Columns check:", "trade_id" in trades.columns)


# ============================================================
# 2. Drawdown Scale
#
# 既に定義済みでも上書きして問題なし
# ============================================================

def drawdown_scale(
    current_drawdown,
    dd_rules
):
    """
    Drawdownの深さによってPosition Sizeを落とす。

    例:
        DD -0.75% -> 0.75倍
        DD -1.25% -> 0.50倍
    """

    dd_magnitude = abs(
        min(
            float(current_drawdown),
            0.0
        )
    )

    scale = 1.0

    for threshold, rule_scale in sorted(dd_rules):

        if dd_magnitude >= threshold:
            scale = rule_scale

    return float(scale)


# ============================================================
# 3. 修正版 Portfolio Simulator
# ============================================================

def simulate_policy(
    data,
    policy_by_year,
    starting_capital=1.0,
):

    """
    Risk Policyを時間順にシミュレーションする。

    改善点:
    row._trade_id を使用しない。
    trade_idという通常列を使う。
    """

    data = data.copy()


    # --------------------------------------------------------
    # 必須列チェック
    # --------------------------------------------------------

    required = [
        "entry_time",
        "exit_time",
        "test_year",
        "position_size",
        "unit_return",
    ]


    missing = [
        col
        for col in required
        if col not in data.columns
    ]


    if missing:
        raise RuntimeError(
            f"simulate_policy に必要な列がありません: {missing}"
        )


    # --------------------------------------------------------
    # trade_idが無ければ自動作成
    # --------------------------------------------------------

    if "trade_id" not in data.columns:

        data = data.copy()

        data["trade_id"] = np.arange(
            len(data),
            dtype=int
        )


    # --------------------------------------------------------
    # 型を安全に統一
    # --------------------------------------------------------

    data["entry_time"] = pd.to_datetime(
        data["entry_time"],
        errors="coerce"
    )

    data["exit_time"] = pd.to_datetime(
        data["exit_time"],
        errors="coerce"
    )


    for col in [
        "position_size",
        "unit_return",
        "test_year",
        "trade_id",
    ]:

        data[col] = pd.to_numeric(
            data[col],
            errors="coerce"
        )


    data = data.dropna(
        subset=[
            "entry_time",
            "exit_time",
            "position_size",
            "unit_return",
            "test_year",
            "trade_id",
        ]
    )


    data["test_year"] = (
        data["test_year"]
        .astype(int)
    )

    data["trade_id"] = (
        data["trade_id"]
        .astype(int)
    )


    data = (
        data
        .sort_values(
            [
                "entry_time",
                "trade_id",
            ]
        )
        .reset_index(drop=True)
    )


    # ========================================================
    # Portfolio State
    # ========================================================

    equity = float(starting_capital)

    high_water = float(starting_capital)

    open_positions = []

    gross_open = 0.0


    current_day = None

    day_start_equity = float(
        starting_capital
    )


    records = []

    equity_records = []


    max_gross_seen = 0.0

    cap_clipped_count = 0

    daily_stopped_count = 0

    dd_reduced_count = 0

    zero_size_count = 0


    # ========================================================
    # Exit処理
    # ========================================================

    def process_exits_until(
        current_time
    ):

        nonlocal equity
        nonlocal high_water
        nonlocal gross_open


        while open_positions:

            next_exit_time = (
                open_positions[0][0]
            )


            if next_exit_time > current_time:
                break


            (
                exit_time,
                trade_id,
                effective_size,
                pnl_amount,
                entry_equity,

            ) = heapq.heappop(
                open_positions
            )


            equity += float(
                pnl_amount
            )


            gross_open -= float(
                effective_size
            )


            gross_open = max(
                0.0,
                gross_open
            )


            high_water = max(
                high_water,
                equity
            )


            equity_records.append({

                "time":
                    exit_time,

                "equity":
                    equity,

                "trade_id":
                    trade_id,

            })


    # ========================================================
    # Entry Loop
    # ========================================================

    #
    # itertuplesを使うが、
    # 今回は trade_id という安全な列名なので
    # row.trade_id で確実に取得できる。
    #

    for row in data.itertuples(
        index=False
    ):


        entry_time = row.entry_time

        exit_time = row.exit_time

        trade_id = int(
            row.trade_id
        )

        year = int(
            row.test_year
        )


        # ----------------------------------------------------
        # Entry以前に終了したPositionを決済
        # ----------------------------------------------------

        process_exits_until(
            entry_time
        )


        # ----------------------------------------------------
        # 日付変更
        # ----------------------------------------------------

        current_trade_day = (
            entry_time.date()
        )


        if current_day != current_trade_day:

            current_day = (
                current_trade_day
            )

            day_start_equity = (
                equity
            )


        # ----------------------------------------------------
        # Risk Policy取得
        # ----------------------------------------------------

        policy_name = (
            policy_by_year.get(
                year,
                "NO_RISK"
            )
        )


        if policy_name not in RISK_POLICIES:

            raise RuntimeError(
                f"未知のRisk Policy: {policy_name}"
            )


        policy = (
            RISK_POLICIES[
                policy_name
            ]
        )


        original_size = max(
            0.0,
            float(
                row.position_size
            )
        )


        desired_size = (
            original_size
        )


        # ====================================================
        # Drawdown
        # ====================================================

        if high_water > 0:

            current_dd = (
                equity
                /
                high_water
                -
                1.0
            )

        else:

            current_dd = 0.0


        dd_scale = drawdown_scale(

            current_dd,

            policy.get(
                "dd_rules",
                []
            )

        )


        if dd_scale < 1.0:
            dd_reduced_count += 1


        desired_size *= (
            dd_scale
        )


        # ====================================================
        # Daily Stop
        # ====================================================

        daily_block = False


        daily_loss_limit = (
            policy.get(
                "daily_loss_limit"
            )
        )


        if (
            daily_loss_limit
            is not None

            and
            day_start_equity > 0
        ):


            daily_return = (
                equity
                /
                day_start_equity
                -
                1.0
            )


            if (
                daily_return
                <=
                -float(
                    daily_loss_limit
                )
            ):

                desired_size = 0.0

                daily_block = True

                daily_stopped_count += 1


        # ====================================================
        # Gross Exposure Cap
        # ====================================================

        gross_cap = (
            policy.get(
                "gross_cap"
            )
        )


        cap_clipped = False


        if gross_cap is not None:

            available_exposure = max(

                0.0,

                float(gross_cap)
                -
                gross_open

            )


            if (
                desired_size
                >
                available_exposure
            ):

                desired_size = (
                    available_exposure
                )

                cap_clipped = True

                cap_clipped_count += 1


        effective_size = max(
            0.0,
            float(
                desired_size
            )
        )


        if effective_size <= EPS:
            zero_size_count += 1


        # ====================================================
        # PnL
        # ====================================================

        entry_equity = float(
            equity
        )


        unit_return = float(
            row.unit_return
        )


        trade_return = (
            effective_size
            *
            unit_return
        )


        pnl_amount = (
            entry_equity
            *
            trade_return
        )


        gross_before = float(
            gross_open
        )


        # ====================================================
        # Position Open
        # ====================================================

        if effective_size > EPS:


            heapq.heappush(

                open_positions,

                (

                    exit_time,

                    trade_id,

                    effective_size,

                    pnl_amount,

                    entry_equity,

                )

            )


            gross_open += (
                effective_size
            )


            max_gross_seen = max(

                max_gross_seen,

                gross_open

            )


        # ====================================================
        # Record
        # ====================================================

        records.append({

            "trade_id":
                trade_id,

            "entry_time":
                entry_time,

            "exit_time":
                exit_time,

            "test_year":
                year,

            "policy":
                policy_name,

            "original_size":
                original_size,

            "effective_size":
                effective_size,

            "dd_scale":
                dd_scale,

            "current_dd":
                current_dd,

            "gross_before":
                gross_before,

            "gross_after":
                gross_open,

            "cap_clipped":
                cap_clipped,

            "daily_block":
                daily_block,

            "unit_return":
                unit_return,

            "trade_return":
                trade_return,

            "pnl_amount":
                pnl_amount,

        })


    # ========================================================
    # 最後まで残ったPositionを全部決済
    # ========================================================

    while open_positions:


        (
            exit_time,
            trade_id,
            effective_size,
            pnl_amount,
            entry_equity,

        ) = heapq.heappop(
            open_positions
        )


        equity += float(
            pnl_amount
        )


        gross_open -= float(
            effective_size
        )


        gross_open = max(
            0.0,
            gross_open
        )


        high_water = max(
            high_water,
            equity
        )


        equity_records.append({

            "time":
                exit_time,

            "equity":
                equity,

            "trade_id":
                trade_id,

        })


    # ========================================================
    # DataFrame化
    # ========================================================

    trade_records = pd.DataFrame(
        records
    )


    equity_df = pd.DataFrame(
        equity_records
    )


    if not equity_df.empty:


        equity_df["time"] = pd.to_datetime(
            equity_df["time"]
        )


        equity_df = (

            equity_df

            .sort_values(
                [
                    "time",
                    "trade_id",
                ]
            )

            .drop_duplicates(
                subset="time",
                keep="last"
            )

            .set_index(
                "time"
            )

        )


    # ========================================================
    # Summary
    # ========================================================

    summary = {

        "starting_capital":
            float(
                starting_capital
            ),

        "ending_capital":
            float(
                equity
            ),

        "growth":
            float(
                equity
                /
                starting_capital
                -
                1.0
            ),

        "max_gross_exposure":
            float(
                max_gross_seen
            ),

        "cap_clipped_count":
            int(
                cap_clipped_count
            ),

        "daily_stopped_count":
            int(
                daily_stopped_count
            ),

        "dd_reduced_count":
            int(
                dd_reduced_count
            ),

        "zero_size_count":
            int(
                zero_size_count
            ),

    }


    return (
        trade_records,
        equity_df,
        summary
    )


# ============================================================
# 4. SIMULATOR TEST
#
# 本番の巨大ループを回す前に
# まずNO_RISKで動作確認する。
# ============================================================

print()
print("=" * 90)
print("SIMULATOR SANITY TEST")
print("=" * 90)


test_policy_mapping = {

    int(year):
        "NO_RISK"

    for year in sorted(
        trades[
            "test_year"
        ].unique()
    )

}


test_rec, test_eq, test_summary = simulate_policy(

    trades,

    test_policy_mapping,

    starting_capital=
        INITIAL_CAPITAL,

)


print(
    "Simulation succeeded."
)

print(
    "Trade records:",
    len(test_rec)
)

print(
    "Equity records:",
    len(test_eq)
)

print(
    "Ending Capital:",
    test_summary[
        "ending_capital"
    ]
)

print(
    "Growth:",
    test_summary[
        "growth"
    ]
    *
    100,
    "%"
)

print(
    "Max Gross Exposure:",
    test_summary[
        "max_gross_exposure"
    ]
)

print()
print(
    "HOTFIX COMPLETE"
)

print(
    "ここまでエラーが出なければ、"
)

print(
    "前回コードの STATIC POLICY COMPARISON 以降を実行してください。"
)


## 元セルindex 48


In [ ]:
# ============================================================
# USD/JPY
# RISK ENGINE v1
# EXPANDING META WALK-FORWARD TEST
#
# 目的
# ------------------------------------------------------------
# 現在完成している
#
#   ML
#   -> Calibration
#   -> Threshold
#   -> Session
#   -> 30min Exit
#   -> Adaptive Position Sizing
#
# を変更せず、
#
#   ・総Exposure上限
#   ・Drawdown時のSize縮小
#   ・Daily Loss Limit
#
# を追加したとき、
#
# リスク調整後OOS成績が改善するかを検証する。
#
#
# 重要
# ------------------------------------------------------------
# Risk Policyも未来情報を使って選ばない。
#
# 例:
#
# 2024 Test
#
# Policy選択:
#   2020-2023のみ使用
#
# Test:
#   2024
#
# ============================================================


from pathlib import Path
from datetime import timedelta, datetime
import heapq

import numpy as np
import pandas as pd


# ============================================================
# 0. CONFIG
# ============================================================

INITIAL_CAPITAL = 1.0

HOLD_MINUTES = 30

BOOTSTRAP_ITERATIONS = 10000

BOOTSTRAP_BLOCK_DAYS = 20

RANDOM_SEED = 42


# ------------------------------------------------------------
# Risk Policy候補
#
# 結果を見た後で数値を微調整しない。
# まずこの固定候補でOOS検証する。
# ------------------------------------------------------------

RISK_POLICIES = {

    # 現在の戦略そのまま
    "NO_RISK": {
        "gross_cap": None,
        "daily_loss_limit": None,
        "dd_rules": [],
    },

    # 同時Exposure最大2.0
    "CAP_2_0": {
        "gross_cap": 2.0,
        "daily_loss_limit": None,
        "dd_rules": [],
    },

    # やや厳しいExposure制限
    "CAP_1_5": {
        "gross_cap": 1.5,
        "daily_loss_limit": None,
        "dd_rules": [],
    },

    # Drawdownが増えたらPosition縮小
    "DD_SOFT": {
        "gross_cap": None,
        "daily_loss_limit": None,

        # DD 0.75% -> 75%
        # DD 1.25% -> 50%
        "dd_rules": [
            (0.0075, 0.75),
            (0.0125, 0.50),
        ],
    },

    # 1日-0.40%で、その日の新規取引停止
    "DAILY_STOP": {
        "gross_cap": None,
        "daily_loss_limit": 0.0040,
        "dd_rules": [],
    },

    # 全部を組み合わせたModerate設定
    "COMBINED": {
        "gross_cap": 2.0,

        "daily_loss_limit": 0.0040,

        "dd_rules": [
            (0.0075, 0.75),
            (0.0125, 0.50),
        ],
    },

}


# ------------------------------------------------------------
# Risk Policy選択Score
#
# 単純利益ではなく、
#
# Growth
# - MaxDD
# - Tail Risk
#
# を使う。
# ------------------------------------------------------------

MAX_DD_PENALTY = 1.00

CVAR_PENALTY = 0.50


# ============================================================
# 1. DATA CHECK
# ============================================================

if "all_test_trades" not in globals():

    raise RuntimeError(
        "all_test_trades がありません。\n"
        "Position Sizingのコードを先に実行してください。"
    )


trades = all_test_trades.copy()


required_columns = [

    "test_year",
    "position_size",
    "sized_return",

]


missing = [

    col

    for col in required_columns

    if col not in trades.columns

]


if missing:

    raise RuntimeError(

        "all_test_trades に必要な列がありません。\n"
        f"不足列: {missing}"

    )


# ============================================================
# 2. TIMESTAMP
# ============================================================

# indexがDatetimeIndexならそのまま使用
if isinstance(
    trades.index,
    pd.DatetimeIndex
):

    trades["entry_time"] = trades.index


# timestamp列がある場合
elif "timestamp" in trades.columns:

    trades["entry_time"] = pd.to_datetime(
        trades["timestamp"],
        errors="coerce"
    )


else:

    # indexをtimestampとして試す
    trades["entry_time"] = pd.to_datetime(
        trades.index,
        errors="coerce"
    )


if trades["entry_time"].isna().any():

    raise RuntimeError(
        "timestampを正しく取得できない行があります。"
    )


trades = trades.sort_values(
    "entry_time"
).reset_index(
    drop=True
)


trades["_trade_id"] = np.arange(
    len(trades)
)


# ============================================================
# 3. NUMERIC CLEANING
# ============================================================

for col in [

    "position_size",
    "sized_return",

]:

    trades[col] = pd.to_numeric(
        trades[col],
        errors="coerce"
    )


trades["test_year"] = pd.to_numeric(
    trades["test_year"],
    errors="coerce"
)


trades = trades.dropna(
    subset=[
        "entry_time",
        "test_year",
        "position_size",
        "sized_return",
    ]
)


trades["test_year"] = (
    trades["test_year"]
    .astype(int)
)


# ============================================================
# 4. UNIT RETURN
# ============================================================

# Adaptive Sizing後Return:
#
# sized_return
#
# を基準にする。
#
#
# unit_return =
#
# sized_return / position_size
#
#
# これでRisk Engine側がSizeを変えた場合も
# 同じ元戦略Returnを利用できる。
# ============================================================

EPS = 1e-12


trades["unit_return"] = np.where(

    trades["position_size"].abs() > EPS,

    trades["sized_return"]
    /
    trades["position_size"],

    0.0,

)


trades["exit_time"] = (

    trades["entry_time"]

    +

    pd.Timedelta(
        minutes=HOLD_MINUTES
    )

)


# ============================================================
# 5. SANITY CHECK
# ============================================================

reconstructed = (

    trades["unit_return"]

    *

    trades["position_size"]

)


reconstruction_mae = float(

    np.mean(

        np.abs(

            reconstructed
            -
            trades["sized_return"]

        )

    )

)


print()
print("=" * 90)

print(
    "DATA CHECK"
)

print("=" * 90)

print(
    "Trades:",
    len(trades)
)

print(
    "Years:",
    sorted(
        trades["test_year"].unique()
    )
)

print(
    "Return reconstruction MAE:",
    reconstruction_mae
)


# ============================================================
# 6. RISK POLICY
# ============================================================

def drawdown_scale(
    current_drawdown,
    dd_rules
):

    """
    現在のDrawdownに応じたPosition Size倍率。

    current_drawdown:
        -0.01 = -1%

    dd_rules:
        [(0.0075, 0.75), ...]
    """

    dd_magnitude = abs(
        min(
            current_drawdown,
            0.0
        )
    )


    scale = 1.0


    for threshold, rule_scale in sorted(
        dd_rules
    ):

        if dd_magnitude >= threshold:

            scale = rule_scale


    return scale


# ============================================================
# 7. PORTFOLIO SIMULATOR
# ============================================================

def simulate_policy(
    data,
    policy_by_year,
    starting_capital=1.0,
):

    """
    実際の時間順にPositionを処理する。

    重要:
    30分保有なので、
    15分後に次のSignalが来れば
    Positionは重複する。

    gross_capはその重複Exposureを制限する。
    """

    data = (

        data
        .sort_values("entry_time")
        .copy()

    )


    equity = float(
        starting_capital
    )


    high_water = equity


    # --------------------------------------------------------
    # Open Position
    #
    # Heap:
    #
    # (
    #   exit_time,
    #   trade_id,
    #   effective_size,
    #   pnl_amount,
    #   entry_equity
    # )
    # --------------------------------------------------------

    open_positions = []


    gross_open = 0.0


    current_day = None

    day_start_equity = equity


    records = []

    equity_records = []


    max_gross_seen = 0.0


    cap_clipped_count = 0

    daily_stopped_count = 0

    dd_reduced_count = 0

    zero_size_count = 0


    # ========================================================
    # Helper: Position Exit
    # ========================================================

    def process_exits_until(
        current_time
    ):

        nonlocal equity
        nonlocal high_water
        nonlocal gross_open


        while (

            open_positions

            and

            open_positions[0][0]
            <=
            current_time

        ):


            (
                exit_time,
                trade_id,
                effective_size,
                pnl_amount,
                entry_equity,

            ) = heapq.heappop(
                open_positions
            )


            equity += pnl_amount


            gross_open -= effective_size


            gross_open = max(
                0.0,
                gross_open
            )


            high_water = max(
                high_water,
                equity
            )


            equity_records.append({

                "time":
                    exit_time,

                "equity":
                    equity,

            })


    # ========================================================
    # Entry loop
    # ========================================================

    for row in data.itertuples(
        index=False
    ):


        entry_time = row.entry_time

        exit_time = row.exit_time

        year = int(
            row.test_year
        )


        # ----------------------------------------------------
        # まず既にExitしたPositionを精算
        # ----------------------------------------------------

        process_exits_until(
            entry_time
        )


        # ----------------------------------------------------
        # 日付変更
        # ----------------------------------------------------

        day = entry_time.date()


        if current_day != day:

            current_day = day

            day_start_equity = equity


        # ----------------------------------------------------
        # Policy
        # ----------------------------------------------------

        policy_name = policy_by_year.get(
            year,
            "NO_RISK"
        )


        policy = RISK_POLICIES[
            policy_name
        ]


        desired_size = float(
            row.position_size
        )


        original_size = desired_size


        # ----------------------------------------------------
        # Drawdown
        # ----------------------------------------------------

        if high_water > 0:

            current_dd = (

                equity
                /
                high_water

                -
                1.0

            )

        else:

            current_dd = 0.0


        dd_scale = drawdown_scale(

            current_dd,

            policy[
                "dd_rules"
            ],

        )


        if dd_scale < 1.0:

            dd_reduced_count += 1


        desired_size *= dd_scale


        # ----------------------------------------------------
        # Daily Loss Limit
        # ----------------------------------------------------

        daily_loss_limit = policy[
            "daily_loss_limit"
        ]


        daily_block = False


        if (
            daily_loss_limit is not None

            and

            day_start_equity > 0
        ):


            daily_return = (

                equity
                /
                day_start_equity

                -
                1.0

            )


            if daily_return <= -daily_loss_limit:

                desired_size = 0.0

                daily_block = True

                daily_stopped_count += 1


        # ----------------------------------------------------
        # Gross Exposure Cap
        # ----------------------------------------------------

        gross_cap = policy[
            "gross_cap"
        ]


        cap_clipped = False


        if gross_cap is not None:


            available = max(

                0.0,

                gross_cap
                -
                gross_open,

            )


            if desired_size > available:

                desired_size = available

                cap_clipped = True

                cap_clipped_count += 1


        effective_size = max(
            0.0,
            desired_size
        )


        if effective_size <= EPS:

            zero_size_count += 1


        # ----------------------------------------------------
        # Entry Equity
        # ----------------------------------------------------

        entry_equity = equity


        # ----------------------------------------------------
        # Future PnL
        #
        # 決定時にはfuture returnは使っていない。
        #
        # Exit PnL計算のためだけに使用する。
        # ----------------------------------------------------

        pnl_amount = (

            entry_equity

            *

            effective_size

            *

            float(
                row.unit_return
            )

        )


        # ----------------------------------------------------
        # Position登録
        # ----------------------------------------------------

        if effective_size > EPS:


            heapq.heappush(

                open_positions,

                (

                    exit_time,

                    int(
                        row._trade_id
                    ),

                    effective_size,

                    pnl_amount,

                    entry_equity,

                ),

            )


            gross_open += effective_size


            max_gross_seen = max(

                max_gross_seen,

                gross_open,

            )


        # ----------------------------------------------------
        # Trade Record
        # ----------------------------------------------------

        records.append({

            "trade_id":
                int(
                    row._trade_id
                ),

            "entry_time":
                entry_time,

            "exit_time":
                exit_time,

            "test_year":
                year,

            "policy":
                policy_name,

            "original_size":
                original_size,

            "effective_size":
                effective_size,

            "dd_scale":
                dd_scale,

            "current_dd":
                current_dd,

            "gross_before":
                gross_open
                -
                effective_size,

            "gross_after":
                gross_open,

            "cap_clipped":
                cap_clipped,

            "daily_block":
                daily_block,

            "unit_return":
                float(
                    row.unit_return
                ),

            "trade_return":
                (
                    effective_size
                    *
                    float(
                        row.unit_return
                    )
                ),

            "pnl_amount":
                pnl_amount,

        })


    # ========================================================
    # 最後のPositionを全部Exit
    # ========================================================

    process_exits_until(
        pd.Timestamp.max.tz_localize(
            None
        )
        if data["exit_time"].dt.tz is None
        else
        data["exit_time"].max()
        +
        pd.Timedelta(
            days=1
        )
    )


    trade_records = pd.DataFrame(
        records
    )


    equity_df = pd.DataFrame(
        equity_records
    )


    if not equity_df.empty:

        equity_df = (

            equity_df

            .sort_values(
                "time"
            )

            .drop_duplicates(
                subset="time",
                keep="last"
            )

            .set_index(
                "time"
            )

        )


    summary = {

        "starting_capital":
            starting_capital,

        "ending_capital":
            equity,

        "growth":
            (
                equity
                /
                starting_capital

                -
                1.0
            ),

        "max_gross_exposure":
            max_gross_seen,

        "cap_clipped_count":
            cap_clipped_count,

        "daily_stopped_count":
            daily_stopped_count,

        "dd_reduced_count":
            dd_reduced_count,

        "zero_size_count":
            zero_size_count,

    }


    return (
        trade_records,
        equity_df,
        summary
    )


# ============================================================
# 8. EQUITY -> DAILY RETURN
# ============================================================

def make_daily_equity(
    equity_df,
    starting_capital=1.0
):

    if equity_df.empty:

        return pd.Series(
            dtype=float
        )


    eq = equity_df[
        "equity"
    ].copy()


    start_day = (
        eq.index.min()
        .normalize()
    )


    end_day = (
        eq.index.max()
        .normalize()
    )


    daily_index = pd.date_range(

        start=start_day,

        end=end_day,

        freq="1D",

        tz=eq.index.tz,

    )


    daily_equity = (

        eq

        .resample(
            "1D"
        )

        .last()

        .reindex(
            daily_index
        )

        .ffill()

    )


    daily_equity.iloc[0] = (

        daily_equity.iloc[0]

        if np.isfinite(
            daily_equity.iloc[0]
        )

        else starting_capital

    )


    daily_returns = (

        daily_equity

        .pct_change()

        .fillna(
            0.0
        )

    )


    return pd.DataFrame({

        "equity":
            daily_equity,

        "daily_return":
            daily_returns,

    })


# ============================================================
# 9. PERFORMANCE STATS
# ============================================================

def calc_performance(
    trade_records,
    equity_df,
    starting_capital=1.0
):


    if trade_records.empty:

        return {}


    r = (

        trade_records[
            "trade_return"
        ]

        .astype(float)

    )


    positive = r[
        r > 0
    ].sum()


    negative = -r[
        r < 0
    ].sum()


    if negative > 0:

        pf = positive / negative

    elif positive > 0:

        pf = np.inf

    else:

        pf = np.nan


    daily = make_daily_equity(

        equity_df,

        starting_capital,

    )


    if daily.empty:

        return {}


    equity = daily[
        "equity"
    ]


    peak = equity.cummax()


    dd = (

        equity
        /
        peak

        -
        1.0

    )


    max_dd = float(
        dd.min()
    )


    growth = float(

        equity.iloc[-1]

        /
        starting_capital

        -
        1.0

    )


    daily_returns = daily[
        "daily_return"
    ]


    worst_day = float(
        daily_returns.min()
    )


    # --------------------------------------------------------
    # Historical CVaR 95%
    #
    # 最悪5%の日の平均損失
    # 正の数字として表示
    # --------------------------------------------------------

    q05 = daily_returns.quantile(
        0.05
    )


    worst_tail = daily_returns[
        daily_returns <= q05
    ]


    if len(
        worst_tail
    ) > 0:

        cvar_95 = float(
            -worst_tail.mean()
        )

    else:

        cvar_95 = np.nan


    if max_dd < 0:

        return_to_dd = (

            growth
            /
            abs(
                max_dd
            )

        )

    else:

        return_to_dd = np.nan


    if daily_returns.std(
        ddof=1
    ) > 0:

        daily_sharpe = (

            daily_returns.mean()

            /
            daily_returns.std(
                ddof=1
            )

            *

            np.sqrt(
                252
            )

        )

    else:

        daily_sharpe = np.nan


    return {

        "trades":
            len(
                trade_records
            ),

        "active_trades":
            int(
                (
                    trade_records[
                        "effective_size"
                    ]
                    >
                    EPS
                ).sum()
            ),

        "mean_effective_size":
            float(
                trade_records[
                    "effective_size"
                ].mean()
            ),

        "win_rate":
            float(
                (
                    r > 0
                ).mean()
            ),

        "avg_trade_return":
            float(
                r.mean()
            ),

        "profit_factor":
            float(
                pf
            ),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            float(
                return_to_dd
            ),

        "daily_sharpe":
            float(
                daily_sharpe
            ),

        "worst_day":
            worst_day,

        "daily_cvar_95":
            cvar_95,

    }


# ============================================================
# 10. POLICY SCORE
# ============================================================

def policy_score(
    stats
):

    if not stats:

        return -np.inf


    growth = stats[
        "growth"
    ]


    max_dd = abs(
        stats[
            "max_dd"
        ]
    )


    cvar = stats[
        "daily_cvar_95"
    ]


    pf = stats[
        "profit_factor"
    ]


    if not np.isfinite(
        growth
    ):

        return -np.inf


    if not np.isfinite(
        pf
    ):

        return -np.inf


    # Edge自体が消えているPolicyは採用しない
    if pf <= 1.0:

        return -np.inf


    score = (

        growth

        -

        MAX_DD_PENALTY
        *
        max_dd

        -

        CVAR_PENALTY
        *
        cvar

    )


    return float(
        score
    )


# ============================================================
# 11. STATIC POLICY TEST
#
# 診断用。
#
# 全期間から一番を選んではいけない。
# ============================================================

print()
print("=" * 90)

print(
    "STATIC POLICY COMPARISON"
)

print(
    "(DIAGNOSTIC ONLY - NOT USED FOR FINAL SELECTION)"
)

print("=" * 90)


static_rows = []


for policy_name in RISK_POLICIES:


    mapping = {

        int(year):
            policy_name

        for year in trades[
            "test_year"
        ].unique()

    }


    rec, eq, sim_summary = simulate_policy(

        trades,

        mapping,

        starting_capital=
            INITIAL_CAPITAL,

    )


    stats = calc_performance(

        rec,

        eq,

        INITIAL_CAPITAL,

    )


    static_rows.append({

        "policy":
            policy_name,

        **stats,

        **sim_summary,

        "score":
            policy_score(
                stats
            ),

    })


static_results = pd.DataFrame(
    static_rows
)


static_show = static_results.copy()


for col in [

    "win_rate",
    "avg_trade_return",
    "growth",
    "max_dd",
    "worst_day",
    "daily_cvar_95",

]:

    if col in static_show.columns:

        static_show[
            col
        ] *= 100


print(

    static_show[
        [

            "policy",
            "growth",
            "max_dd",
            "profit_factor",
            "return_to_dd",
            "daily_sharpe",
            "worst_day",
            "daily_cvar_95",
            "max_gross_exposure",
            "cap_clipped_count",
            "daily_stopped_count",
            "dd_reduced_count",
            "score",

        ]
    ]

    .sort_values(
        "score",
        ascending=False
    )

    .to_string(
        index=False
    )

)


# ============================================================
# 12. EXPANDING WALK-FORWARD POLICY SELECTION
# ============================================================

years = sorted(
    trades[
        "test_year"
    ].unique()
)


if len(
    years
) < 2:

    raise RuntimeError(
        "Risk Walk-Forwardには最低2年必要です。"
    )


warmup_year = years[0]


evaluation_years = years[
    1:
]


print()
print("=" * 90)

print(
    "EXPANDING META WALK-FORWARD POLICY SELECTION"
)

print("=" * 90)


print(
    "Warm-up year:",
    warmup_year
)

print(
    "Evaluation years:",
    evaluation_years
)


selection_rows = []

selected_policy_by_year = {}


# Warm-up yearはRiskなし
selected_policy_by_year[
    warmup_year
] = "NO_RISK"


for test_year in evaluation_years:


    historical = trades.loc[

        trades[
            "test_year"
        ]
        <
        test_year

    ].copy()


    candidate_rows = []


    for policy_name in RISK_POLICIES:


        mapping = {

            int(year):
                policy_name

            for year in historical[
                "test_year"
            ].unique()

        }


        rec, eq, sim_summary = simulate_policy(

            historical,

            mapping,

            starting_capital=
                INITIAL_CAPITAL,

        )


        stats = calc_performance(

            rec,

            eq,

            INITIAL_CAPITAL,

        )


        score = policy_score(
            stats
        )


        candidate_rows.append({

            "test_year":
                int(
                    test_year
                ),

            "policy":
                policy_name,

            "history_start":
                int(
                    historical[
                        "test_year"
                    ].min()
                ),

            "history_end":
                int(
                    historical[
                        "test_year"
                    ].max()
                ),

            "history_trades":
                len(
                    historical
                ),

            "score":
                score,

            "growth":
                stats[
                    "growth"
                ],

            "max_dd":
                stats[
                    "max_dd"
                ],

            "pf":
                stats[
                    "profit_factor"
                ],

            "return_to_dd":
                stats[
                    "return_to_dd"
                ],

            "cvar_95":
                stats[
                    "daily_cvar_95"
                ],

        })


    candidate_table = pd.DataFrame(
        candidate_rows
    )


    candidate_table = candidate_table.sort_values(

        [
            "score",
            "pf",
        ],

        ascending=[
            False,
            False,
        ],

    )


    winner = candidate_table.iloc[
        0
    ]


    chosen_policy = winner[
        "policy"
    ]


    selected_policy_by_year[
        int(
            test_year
        )
    ] = chosen_policy


    selection_rows.extend(
        candidate_rows
    )


    print()
    print("-" * 70)

    print(
        "TEST YEAR:",
        test_year
    )

    print(
        "History:",
        int(
            historical[
                "test_year"
            ].min()
        ),
        "->",
        int(
            historical[
                "test_year"
            ].max()
        )
    )

    print(
        "Selected Risk Policy:",
        chosen_policy
    )

    print(
        candidate_table[
            [
                "policy",
                "score",
                "growth",
                "max_dd",
                "pf",
                "return_to_dd",
                "cvar_95",
            ]
        ]
        .head(
            len(
                RISK_POLICIES
            )
        )
        .to_string(
            index=False
        )
    )


selection_results = pd.DataFrame(
    selection_rows
)


# ============================================================
# 13. FINAL META-OOS DATA
# ============================================================

# Warm-up年は最終評価から除く
meta_data = trades.loc[

    trades[
        "test_year"
    ].isin(
        evaluation_years
    )

].copy()


# ============================================================
# 14. BASELINE
#
# Current Adaptive Position Sizing
# ============================================================

baseline_mapping = {

    int(year):
        "NO_RISK"

    for year in evaluation_years

}


baseline_records, baseline_eq, baseline_summary = (

    simulate_policy(

        meta_data,

        baseline_mapping,

        starting_capital=
            INITIAL_CAPITAL,

    )

)


baseline_stats = calc_performance(

    baseline_records,

    baseline_eq,

    INITIAL_CAPITAL,

)


# ============================================================
# 15. META WALK-FORWARD RISK ENGINE
# ============================================================

risk_mapping = {

    int(year):
        selected_policy_by_year[
            int(year)
        ]

    for year in evaluation_years

}


risk_records, risk_eq, risk_summary = simulate_policy(

    meta_data,

    risk_mapping,

    starting_capital=
        INITIAL_CAPITAL,

)


risk_stats = calc_performance(

    risk_records,

    risk_eq,

    INITIAL_CAPITAL,

)


# ============================================================
# 16. OVERALL RESULT
# ============================================================

print()
print("=" * 90)

print(
    "META WALK-FORWARD OOS RESULT"
)

print("=" * 90)


overall = pd.DataFrame({

    "BASELINE":
        {
            **baseline_stats,
            **baseline_summary,
        },

    "RISK_ENGINE":
        {
            **risk_stats,
            **risk_summary,
        },

}).T


overall_show = overall.copy()


for col in [

    "win_rate",
    "avg_trade_return",
    "growth",
    "max_dd",
    "worst_day",
    "daily_cvar_95",

]:

    if col in overall_show.columns:

        overall_show[
            col
        ] *= 100


print(
    overall_show.to_string()
)


# ============================================================
# 17. YEARLY META-OOS RESULT
# ============================================================

print()
print("=" * 90)

print(
    "YEARLY META-OOS RESULT"
)

print("=" * 90)


annual_rows = []


for year in evaluation_years:


    year_data = trades.loc[

        trades[
            "test_year"
        ]
        ==
        year

    ].copy()


    # Baseline
    b_map = {
        int(year):
            "NO_RISK"
    }


    b_rec, b_eq, b_sum = simulate_policy(

        year_data,

        b_map,

        INITIAL_CAPITAL,

    )


    b_stats = calc_performance(

        b_rec,

        b_eq,

        INITIAL_CAPITAL,

    )


    # Risk
    policy_name = selected_policy_by_year[
        int(year)
    ]


    r_map = {
        int(year):
            policy_name
    }


    r_rec, r_eq, r_sum = simulate_policy(

        year_data,

        r_map,

        INITIAL_CAPITAL,

    )


    r_stats = calc_performance(

        r_rec,

        r_eq,

        INITIAL_CAPITAL,

    )


    annual_rows.append({

        "test_year":
            int(year),

        "selected_policy":
            policy_name,

        "base_growth":
            b_stats[
                "growth"
            ],

        "risk_growth":
            r_stats[
                "growth"
            ],

        "base_pf":
            b_stats[
                "profit_factor"
            ],

        "risk_pf":
            r_stats[
                "profit_factor"
            ],

        "base_max_dd":
            b_stats[
                "max_dd"
            ],

        "risk_max_dd":
            r_stats[
                "max_dd"
            ],

        "base_return_dd":
            b_stats[
                "return_to_dd"
            ],

        "risk_return_dd":
            r_stats[
                "return_to_dd"
            ],

        "base_cvar":
            b_stats[
                "daily_cvar_95"
            ],

        "risk_cvar":
            r_stats[
                "daily_cvar_95"
            ],

        "cap_clipped":
            r_sum[
                "cap_clipped_count"
            ],

        "daily_stopped":
            r_sum[
                "daily_stopped_count"
            ],

        "dd_reduced":
            r_sum[
                "dd_reduced_count"
            ],

    })


annual_results = pd.DataFrame(
    annual_rows
)


annual_show = annual_results.copy()


for col in [

    "base_growth",
    "risk_growth",
    "base_max_dd",
    "risk_max_dd",
    "base_cvar",
    "risk_cvar",

]:

    annual_show[
        col
    ] *= 100


print(
    annual_show.to_string(
        index=False
    )
)


# ============================================================
# 18. POLICY SELECTION FREQUENCY
# ============================================================

print()
print("=" * 90)

print(
    "RISK POLICY SELECTION FREQUENCY"
)

print("=" * 90)


selection_frequency = pd.Series(
    risk_mapping
).value_counts()


print(
    selection_frequency
)


# ============================================================
# 19. DAILY RETURN ALIGNMENT
# ============================================================

baseline_daily = make_daily_equity(

    baseline_eq,

    INITIAL_CAPITAL,

)


risk_daily = make_daily_equity(

    risk_eq,

    INITIAL_CAPITAL,

)


daily_compare = pd.concat(

    [

        baseline_daily[
            "daily_return"
        ].rename(
            "baseline_return"
        ),

        risk_daily[
            "daily_return"
        ].rename(
            "risk_return"
        ),

    ],

    axis=1,

).fillna(
    0.0
)


daily_compare[
    "delta"
] = (

    daily_compare[
        "risk_return"
    ]

    -

    daily_compare[
        "baseline_return"
    ]

)


# ============================================================
# 20. MOVING BLOCK BOOTSTRAP
# ============================================================

def moving_block_bootstrap(
    values,
    block_size=20,
    iterations=10000,
    seed=42,
):


    x = np.asarray(
        values,
        dtype=float
    )


    x = x[
        np.isfinite(
            x
        )
    ]


    n = len(
        x
    )


    if n < block_size:

        return {

            "observed":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "prob_positive":
                np.nan,

        }


    rng = np.random.default_rng(
        seed
    )


    samples = []


    blocks_needed = int(

        np.ceil(
            n
            /
            block_size
        )

    )


    max_start = (

        n
        -
        block_size

    )


    for _ in range(
        iterations
    ):


        pieces = []


        for __ in range(
            blocks_needed
        ):


            start = int(

                rng.integers(

                    0,

                    max_start
                    +
                    1

                )

            )


            pieces.append(

                x[
                    start
                    :
                    start
                    +
                    block_size
                ]

            )


        sample = np.concatenate(
            pieces
        )[:n]


        samples.append(
            np.mean(
                sample
            )
        )


    samples = np.asarray(
        samples
    )


    return {

        "observed":
            float(
                np.mean(x)
            ),

        "ci_low":
            float(
                np.percentile(
                    samples,
                    2.5
                )
            ),

        "ci_high":
            float(
                np.percentile(
                    samples,
                    97.5
                )
            ),

        "prob_positive":
            float(
                np.mean(
                    samples > 0
                )
            ),

    }


bootstrap = moving_block_bootstrap(

    daily_compare[
        "delta"
    ],

    block_size=
        BOOTSTRAP_BLOCK_DAYS,

    iterations=
        BOOTSTRAP_ITERATIONS,

    seed=
        RANDOM_SEED,

)


print()
print("=" * 90)

print(
    "RISK ENGINE DAILY RETURN BOOTSTRAP"
)

print("=" * 90)


print(
    "Observed Mean Delta:",
    bootstrap[
        "observed"
    ]
    *
    100,
    "%"
)


print(
    "95% CI:",
    bootstrap[
        "ci_low"
    ]
    *
    100,
    "%",
    "~",
    bootstrap[
        "ci_high"
    ]
    *
    100,
    "%"
)


print(
    "P(Risk Engine Return > Baseline):",
    bootstrap[
        "prob_positive"
    ]
    *
    100,
    "%"
)


# ============================================================
# 21. RISK VALUE DIAGNOSIS
# ============================================================

base_growth = baseline_stats[
    "growth"
]


risk_growth = risk_stats[
    "growth"
]


base_dd = abs(
    baseline_stats[
        "max_dd"
    ]
)


risk_dd = abs(
    risk_stats[
        "max_dd"
    ]
)


base_cvar = baseline_stats[
    "daily_cvar_95"
]


risk_cvar = risk_stats[
    "daily_cvar_95"
]


base_rdd = baseline_stats[
    "return_to_dd"
]


risk_rdd = risk_stats[
    "return_to_dd"
]


# ------------------------------------------------------------
# Growth Retention
# ------------------------------------------------------------

if base_growth > 0:

    growth_retention = (

        risk_growth
        /
        base_growth

    )

else:

    growth_retention = np.nan


# ------------------------------------------------------------
# DD Improvement
# ------------------------------------------------------------

if base_dd > 0:

    dd_improvement = (

        base_dd
        -
        risk_dd

    ) / base_dd

else:

    dd_improvement = np.nan


# ------------------------------------------------------------
# CVaR Improvement
# ------------------------------------------------------------

if base_cvar > 0:

    cvar_improvement = (

        base_cvar
        -
        risk_cvar

    ) / base_cvar

else:

    cvar_improvement = np.nan


# ------------------------------------------------------------
# Return/DD Improvement
# ------------------------------------------------------------

if base_rdd > 0:

    return_dd_improvement = (

        risk_rdd
        /
        base_rdd

        -
        1.0

    )

else:

    return_dd_improvement = np.nan


print()
print("=" * 90)

print(
    "AUTOMATIC RISK DIAGNOSIS"
)

print("=" * 90)


print(
    "Growth Retention:",
    growth_retention
    *
    100,
    "%"
)


print(
    "Max DD Improvement:",
    dd_improvement
    *
    100,
    "%"
)


print(
    "Daily CVaR Improvement:",
    cvar_improvement
    *
    100,
    "%"
)


print(
    "Return/DD Improvement:",
    return_dd_improvement
    *
    100,
    "%"
)


# ============================================================
# 22. YEAR STABILITY
# ============================================================

annual_results[
    "dd_better"
] = (

    annual_results[
        "risk_max_dd"
    ].abs()

    <

    annual_results[
        "base_max_dd"
    ].abs()

)


annual_results[
    "return_dd_better"
] = (

    annual_results[
        "risk_return_dd"
    ]

    >

    annual_results[
        "base_return_dd"
    ]

)


annual_results[
    "growth_positive"
] = (

    annual_results[
        "risk_growth"
    ]

    >
    0

)


print()
print(
    "DD better years:",
    int(
        annual_results[
            "dd_better"
        ].sum()
    ),
    "/",
    len(
        annual_results
    )
)


print(
    "Return/DD better years:",
    int(
        annual_results[
            "return_dd_better"
        ].sum()
    ),
    "/",
    len(
        annual_results
    )
)


print(
    "Positive Growth years:",
    int(
        annual_results[
            "growth_positive"
        ].sum()
    ),
    "/",
    len(
        annual_results
    )
)


# ============================================================
# 23. FINAL DECISION
# ============================================================

print()
print("=" * 90)

print(
    "FINAL DECISION"
)

print("=" * 90)


clear_risk_value = (

    np.isfinite(
        growth_retention
    )

    and

    np.isfinite(
        dd_improvement
    )

    and

    np.isfinite(
        return_dd_improvement
    )

    and

    growth_retention >= 0.90

    and

    dd_improvement >= 0.10

    and

    return_dd_improvement >= 0.10

)


risk_value_with_cost = (

    np.isfinite(
        growth_retention
    )

    and

    np.isfinite(
        dd_improvement
    )

    and

    growth_retention >= 0.80

    and

    dd_improvement >= 0.15

)


if clear_risk_value:


    print(
        "RESULT: RISK ENGINE HAS CLEAR OOS VALUE"
    )

    print()

    print(
        "利益の90%以上を維持しつつ、"
    )

    print(
        "Drawdown / Return-to-DDを改善しています。"
    )

    print()

    print(
        "NEXT:"
    )

    print(
        "Risk Engineを候補として固定し、"
    )

    print(
        "Machine Learning Model Tournamentへ進みます。"
    )


elif risk_value_with_cost:


    print(
        "RESULT: RISK ENGINE REDUCES RISK WITH RETURN COST"
    )

    print()

    print(
        "リターンを多少犠牲にしてRiskを削減しています。"
    )

    print()

    print(
        "採用するかはPaper Trading前に再判断します。"
    )


else:


    print(
        "RESULT: NO CLEAR OOS RISK ENGINE VALUE"
    )

    print()

    print(
        "現在のAdaptive Position Sizingを維持します。"
    )

    print()

    print(
        "Risk Policyのパラメータを結果に合わせて"
    )

    print(
        "再最適化することはしません。"
    )

    print()

    print(
        "NEXT:"
    )

    print(
        "Machine Learning Model Tournamentへ進みます。"
    )


# ============================================================
# 24. SAVE
# ============================================================

OUTPUT_DIR = (

    Path.cwd()

    /

    (
        "risk_engine_v1_"
        +
        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )
    )

)


OUTPUT_DIR.mkdir(
    exist_ok=False
)


static_results.to_csv(

    OUTPUT_DIR
    /
    "static_policy_comparison.csv",

    index=False,

)


selection_results.to_csv(

    OUTPUT_DIR
    /
    "walk_forward_policy_selection.csv",

    index=False,

)


annual_results.to_csv(

    OUTPUT_DIR
    /
    "annual_meta_oos_results.csv",

    index=False,

)


baseline_records.to_csv(

    OUTPUT_DIR
    /
    "baseline_trade_records.csv",

    index=False,

)


risk_records.to_csv(

    OUTPUT_DIR
    /
    "risk_engine_trade_records.csv",

    index=False,

)


daily_compare.to_csv(

    OUTPUT_DIR
    /
    "daily_return_comparison.csv",

)


overall.to_csv(

    OUTPUT_DIR
    /
    "overall_result.csv",

)


print()
print("=" * 90)

print(
    "FINISHED"
)

print("=" * 90)


print(
    OUTPUT_DIR.resolve()
)


print()
print(
    "スクショしてほしい場所:"
)

print(
    "1. STATIC POLICY COMPARISON"
)

print(
    "2. EXPANDING META WALK-FORWARD POLICY SELECTION"
)

print(
    "3. META WALK-FORWARD OOS RESULT"
)

print(
    "4. YEARLY META-OOS RESULT"
)

print(
    "5. RISK POLICY SELECTION FREQUENCY"
)

print(
    "6. RISK ENGINE DAILY RETURN BOOTSTRAP"
)

print(
    "7. AUTOMATIC RISK DIAGNOSIS"
)

print(
    "8. FINAL DECISION"
)


## 元セルindex 49


In [ ]:
# ============================================================
# USD/JPY RISK ENGINE v2
# ONE-CELL COMPLETE VERSION
# ============================================================
#
# 現在のChampion:
#
# 15m
#   ↓
# Random Forest
#   ↓
# Calibration
#   ↓
# Confidence Threshold
#   ↓
# Session Filter
#   ↓
# Fixed 30-minute Exit
#   ↓
# Confidence-based Position Sizing
#   ↓
# ★ Risk Engine ← 今回
#
#
# 今回検証するRisk:
#
# 1. NO_RISK
# 2. Exposure Cap 2.0
# 3. Exposure Cap 1.5
# 4. Drawdown-based reduction
# 5. Daily Loss Stop
# 6. Combined Risk Engine
#
#
# 重要:
#
# Risk設定は未来のTest年を見て選ばない。
#
# 2022をTestするなら
#
# 2020 + 2021
#
# だけを使ってRisk Policyを選ぶ。
#
# ============================================================


# ============================================================
# 0. IMPORT
# ============================================================

from pathlib import Path
from datetime import datetime
import heapq
import warnings

import numpy as np
import pandas as pd


warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 1. CONFIG
# ============================================================

INITIAL_CAPITAL = 1.0

HOLD_MINUTES = 30

EPS = 1e-12

BOOTSTRAP_ITERATIONS = 5000

BOOTSTRAP_BLOCK_DAYS = 20

RANDOM_SEED = 42


# ============================================================
# 2. RISK POLICY
# ============================================================

RISK_POLICIES = {

    # --------------------------------------------------------
    # 現在のAdaptive Sizingそのまま
    # --------------------------------------------------------
    "NO_RISK": {

        "gross_cap": None,

        "daily_loss_limit": None,

        "dd_rules": [],

    },


    # --------------------------------------------------------
    # 同時総Exposureを2.0に制限
    # --------------------------------------------------------
    "CAP_2_0": {

        "gross_cap": 2.0,

        "daily_loss_limit": None,

        "dd_rules": [],

    },


    # --------------------------------------------------------
    # 同時総Exposureを1.5に制限
    # --------------------------------------------------------
    "CAP_1_5": {

        "gross_cap": 1.5,

        "daily_loss_limit": None,

        "dd_rules": [],

    },


    # --------------------------------------------------------
    # Drawdown拡大時にSize縮小
    #
    # DD -0.75% → 75%
    # DD -1.25% → 50%
    # --------------------------------------------------------
    "DD_SOFT": {

        "gross_cap": None,

        "daily_loss_limit": None,

        "dd_rules": [

            (0.0075, 0.75),

            (0.0125, 0.50),

        ],

    },


    # --------------------------------------------------------
    # 1日の実現損益が-0.40%以下なら
    # その日の新規Entry停止
    # --------------------------------------------------------
    "DAILY_STOP": {

        "gross_cap": None,

        "daily_loss_limit": 0.0040,

        "dd_rules": [],

    },


    # --------------------------------------------------------
    # Exposure + DD + Daily Stop
    # --------------------------------------------------------
    "COMBINED": {

        "gross_cap": 2.0,

        "daily_loss_limit": 0.0040,

        "dd_rules": [

            (0.0075, 0.75),

            (0.0125, 0.50),

        ],

    },

}


# ============================================================
# 3. SOURCE DATA AUTO DETECTION
# ============================================================

print()
print("=" * 100)
print("SOURCE DATA DETECTION")
print("=" * 100)


preferred_names = [

    "all_test_trades",

    "position_sizing_trades",

    "all_oos_trades",

    "oos_trades",

]


source_df = None
source_name = None


# ------------------------------------------------------------
# まず既知の変数名から探す
# ------------------------------------------------------------

for name in preferred_names:

    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):

        if len(obj) > 0:

            source_df = obj.copy()

            source_name = name

            break


# ------------------------------------------------------------
# 見つからなければDataFrameを自動探索
# ------------------------------------------------------------

if source_df is None:

    candidates = []

    for name, obj in list(globals().items()):

        if not isinstance(obj, pd.DataFrame):
            continue

        if len(obj) == 0:
            continue

        cols = set(obj.columns)

        has_year = (
            "test_year" in cols
            or
            "year" in cols
        )

        has_size = any(
            c in cols
            for c in [
                "position_size",
                "size",
                "sizing_multiplier",
            ]
        )

        has_return = any(
            c in cols
            for c in [
                "sized_return",
                "adaptive_return",
                "final_return",
                "strategy_return",
                "trade_return",
                "net_return",
            ]
        )

        if (
            has_year
            and
            has_size
            and
            has_return
        ):

            candidates.append(
                (
                    len(obj),
                    name,
                    obj
                )
            )


    if candidates:

        candidates.sort(
            reverse=True,
            key=lambda x: x[0]
        )

        _, source_name, source_df = candidates[0]

        source_df = source_df.copy()


if source_df is None:

    raise RuntimeError(
        "\nPosition Sizingの取引DataFrameを発見できません。\n"
        "Position Sizingの検証コードを先に1回実行してください。\n"
        "通常は all_test_trades が必要です。"
    )


print(
    "Selected source:",
    source_name
)

print(
    "Rows:",
    len(source_df)
)


# ============================================================
# 4. COLUMN DETECTOR
# ============================================================

def find_column(
    df,
    candidates,
    required=True
):

    for col in candidates:

        if col in df.columns:

            return col


    if required:

        raise RuntimeError(

            "\n必要な列を発見できません。\n"
            f"候補: {candidates}\n"
            f"実際のColumns:\n{list(df.columns)}"

        )


    return None


year_col = find_column(

    source_df,

    [
        "test_year",
        "year",
    ]

)


size_col = find_column(

    source_df,

    [
        "position_size",
        "sizing_multiplier",
        "size",
        "bet_size",
        "exposure",
    ]

)


return_col = find_column(

    source_df,

    [
        "sized_return",
        "adaptive_return",
        "final_return",
        "strategy_return",
        "trade_return",
        "net_return",
    ]

)


time_col = find_column(

    source_df,

    [
        "entry_time",
        "timestamp",
        "datetime",
        "time",
        "date",
    ],

    required=False

)


exit_col = find_column(

    source_df,

    [
        "exit_time",
        "close_time",
    ],

    required=False

)


print()
print("Column mapping:")
print("Year   :", year_col)
print("Size   :", size_col)
print("Return :", return_col)
print("Time   :", time_col)
print("Exit   :", exit_col)


# ============================================================
# 5. STANDARDIZE DATA
# ============================================================

trades = pd.DataFrame()


# ------------------------------------------------------------
# Year
# ------------------------------------------------------------

trades["test_year"] = pd.to_numeric(

    source_df[year_col],

    errors="coerce"

)


# ------------------------------------------------------------
# Position Size
# ------------------------------------------------------------

trades["position_size"] = pd.to_numeric(

    source_df[size_col],

    errors="coerce"

)


# Position SizeはExposure倍率なので正値にする
trades["position_size"] = (

    trades["position_size"]

    .abs()

)


# ------------------------------------------------------------
# Sized Return
# ------------------------------------------------------------

trades["sized_return"] = pd.to_numeric(

    source_df[return_col],

    errors="coerce"

)


# ------------------------------------------------------------
# Entry Time
# ------------------------------------------------------------

if time_col is not None:

    entry_time_raw = source_df[time_col]

else:

    if isinstance(
        source_df.index,
        pd.DatetimeIndex
    ):

        entry_time_raw = source_df.index

    else:

        # indexが文字列DateTimeの可能性も試す
        entry_time_raw = source_df.index


trades["entry_time"] = pd.to_datetime(

    entry_time_raw,

    errors="coerce",

    utc=True

)


# ------------------------------------------------------------
# Exit Time
# ------------------------------------------------------------

if exit_col is not None:

    trades["exit_time"] = pd.to_datetime(

        source_df[exit_col],

        errors="coerce",

        utc=True

    )

else:

    trades["exit_time"] = (

        trades["entry_time"]

        +

        pd.Timedelta(
            minutes=HOLD_MINUTES
        )

    )


# ------------------------------------------------------------
# Cleaning
# ------------------------------------------------------------

trades = trades.dropna(

    subset=[

        "test_year",

        "position_size",

        "sized_return",

        "entry_time",

        "exit_time",

    ]

).copy()


trades = trades.loc[

    trades["position_size"] > EPS

].copy()


trades["test_year"] = (

    trades["test_year"]

    .astype(int)

)


# ------------------------------------------------------------
# 普通の安全な列名を使用
#
# _trade_id は絶対使わない
# ------------------------------------------------------------

trades["trade_id"] = np.arange(

    len(trades),

    dtype=int

)


# ------------------------------------------------------------
# Unit Return
#
# sized_return
# =
# unit_return × position_size
# ------------------------------------------------------------

trades["unit_return"] = (

    trades["sized_return"]

    /

    trades["position_size"]

)


trades = (

    trades

    .sort_values(
        [
            "entry_time",
            "trade_id",
        ]
    )

    .reset_index(
        drop=True
    )

)


# ============================================================
# 6. DATA SANITY CHECK
# ============================================================

reconstructed = (

    trades["unit_return"]

    *

    trades["position_size"]

)


reconstruction_mae = float(

    np.mean(

        np.abs(

            reconstructed

            -

            trades["sized_return"]

        )

    )

)


years = sorted(

    trades["test_year"]

    .unique()
    .tolist()

)


print()
print("=" * 100)
print("DATA CHECK")
print("=" * 100)

print(
    "Trades:",
    len(trades)
)

print(
    "Years:",
    years
)

print(
    "Period:",
    trades["entry_time"].min(),
    "->",
    trades["entry_time"].max()
)

print(
    "Return reconstruction MAE:",
    reconstruction_mae
)


if reconstruction_mae > 1e-10:

    print(
        "WARNING: Return reconstruction error is larger than expected."
    )


if len(years) < 2:

    raise RuntimeError(
        "Risk Walk-Forwardには最低2年のOOSデータが必要です。"
    )


# ============================================================
# 7. DRAWDOWN SCALE
# ============================================================

def get_dd_scale(
    current_dd,
    dd_rules
):

    dd_depth = abs(
        min(
            float(current_dd),
            0.0
        )
    )


    scale = 1.0


    for threshold, value in sorted(
        dd_rules
    ):

        if dd_depth >= threshold:

            scale = float(value)


    return scale


# ============================================================
# 8. PORTFOLIO SIMULATOR
# ============================================================

def simulate_policy(
    data,
    policy_by_year,
    starting_capital=1.0
):

    """
    時系列順にEntry / Exitを処理。

    ・30分保有
    ・15分足なのでPosition重複あり
    ・Gross Exposureを管理
    ・DrawdownでSize変更
    ・Daily Loss Stop対応
    """

    df = (

        data

        .sort_values(
            [
                "entry_time",
                "trade_id",
            ]
        )

        .reset_index(
            drop=True
        )

        .copy()

    )


    equity = float(
        starting_capital
    )


    high_water = float(
        starting_capital
    )


    gross_open = 0.0


    # heap structure:
    #
    # (
    #   exit_time,
    #   trade_id,
    #   effective_size,
    #   pnl
    # )
    open_positions = []


    current_day = None

    day_start_equity = float(
        starting_capital
    )


    trade_records = []

    equity_records = []


    max_gross_exposure = 0.0

    cap_clipped = 0

    daily_blocks = 0

    dd_reductions = 0

    zero_entries = 0


    # ========================================================
    # Exit helper
    # ========================================================

    def close_until(
        current_time
    ):

        nonlocal equity
        nonlocal high_water
        nonlocal gross_open


        while open_positions:

            next_exit = (
                open_positions[0][0]
            )


            if next_exit > current_time:

                break


            (
                exit_time,
                trade_id,
                effective_size,
                pnl,

            ) = heapq.heappop(
                open_positions
            )


            equity += float(
                pnl
            )


            gross_open -= float(
                effective_size
            )


            gross_open = max(
                0.0,
                gross_open
            )


            high_water = max(
                high_water,
                equity
            )


            equity_records.append({

                "time":
                    exit_time,

                "trade_id":
                    int(trade_id),

                "equity":
                    float(equity),

            })


    # ========================================================
    # Main loop
    # ========================================================

    for row in df.itertuples(
        index=False
    ):


        entry_time = row.entry_time

        exit_time = row.exit_time

        trade_id = int(
            row.trade_id
        )

        year = int(
            row.test_year
        )


        # ----------------------------------------------------
        # 先にExit
        # ----------------------------------------------------

        close_until(
            entry_time
        )


        # ----------------------------------------------------
        # New day
        # ----------------------------------------------------

        trade_day = (
            entry_time.date()
        )


        if current_day != trade_day:

            current_day = trade_day

            day_start_equity = float(
                equity
            )


        # ----------------------------------------------------
        # Policy
        # ----------------------------------------------------

        policy_name = policy_by_year.get(

            year,

            "NO_RISK"

        )


        if policy_name not in RISK_POLICIES:

            policy_name = "NO_RISK"


        policy = RISK_POLICIES[
            policy_name
        ]


        original_size = float(
            row.position_size
        )


        desired_size = original_size


        # ====================================================
        # Current Drawdown
        # ====================================================

        if high_water > 0:

            current_dd = (

                equity
                /
                high_water

                -
                1.0

            )

        else:

            current_dd = 0.0


        dd_scale = get_dd_scale(

            current_dd,

            policy.get(
                "dd_rules",
                []
            )

        )


        if dd_scale < 1.0:

            dd_reductions += 1


        desired_size *= (
            dd_scale
        )


        # ====================================================
        # Daily Stop
        # ====================================================

        daily_block = False


        daily_loss_limit = policy.get(
            "daily_loss_limit"
        )


        if (
            daily_loss_limit is not None

            and

            day_start_equity > 0
        ):


            current_daily_return = (

                equity
                /
                day_start_equity

                -
                1.0

            )


            if (
                current_daily_return
                <=
                -float(
                    daily_loss_limit
                )
            ):

                desired_size = 0.0

                daily_block = True

                daily_blocks += 1


        # ====================================================
        # Gross Exposure Cap
        # ====================================================

        cap_hit = False


        gross_cap = policy.get(
            "gross_cap"
        )


        if gross_cap is not None:


            available = max(

                0.0,

                float(
                    gross_cap
                )

                -

                gross_open

            )


            if desired_size > available:

                desired_size = available

                cap_hit = True

                cap_clipped += 1


        effective_size = max(

            0.0,

            float(
                desired_size
            )

        )


        if effective_size <= EPS:

            zero_entries += 1


        # ====================================================
        # Return
        # ====================================================

        unit_return = float(
            row.unit_return
        )


        trade_return = (

            effective_size

            *

            unit_return

        )


        # ----------------------------------------------------
        # PnLはEntry時点のEquityを基準
        # ----------------------------------------------------

        pnl = (

            float(
                equity
            )

            *

            trade_return

        )


        gross_before = float(
            gross_open
        )


        # ====================================================
        # Position Open
        # ====================================================

        if effective_size > EPS:


            heapq.heappush(

                open_positions,

                (

                    exit_time,

                    trade_id,

                    effective_size,

                    pnl,

                )

            )


            gross_open += (
                effective_size
            )


            max_gross_exposure = max(

                max_gross_exposure,

                gross_open

            )


        # ====================================================
        # Record
        # ====================================================

        trade_records.append({

            "trade_id":
                trade_id,

            "entry_time":
                entry_time,

            "exit_time":
                exit_time,

            "test_year":
                year,

            "policy":
                policy_name,

            "original_size":
                original_size,

            "effective_size":
                effective_size,

            "current_dd":
                current_dd,

            "dd_scale":
                dd_scale,

            "gross_before":
                gross_before,

            "gross_after":
                gross_open,

            "cap_hit":
                cap_hit,

            "daily_block":
                daily_block,

            "unit_return":
                unit_return,

            "trade_return":
                trade_return,

            "pnl":
                pnl,

        })


    # ========================================================
    # Close all remaining positions
    # ========================================================

    while open_positions:


        (
            exit_time,
            trade_id,
            effective_size,
            pnl,

        ) = heapq.heappop(
            open_positions
        )


        equity += float(
            pnl
        )


        gross_open -= float(
            effective_size
        )


        gross_open = max(
            0.0,
            gross_open
        )


        high_water = max(
            high_water,
            equity
        )


        equity_records.append({

            "time":
                exit_time,

            "trade_id":
                int(trade_id),

            "equity":
                float(equity),

        })


    trade_df = pd.DataFrame(
        trade_records
    )


    equity_df = pd.DataFrame(
        equity_records
    )


    if not equity_df.empty:


        equity_df = (

            equity_df

            .sort_values(
                [
                    "time",
                    "trade_id",
                ]
            )

            .drop_duplicates(
                subset="time",
                keep="last"
            )

            .set_index(
                "time"
            )

        )


    summary = {

        "starting_capital":
            starting_capital,

        "ending_capital":
            float(
                equity
            ),

        "growth":
            float(
                equity
                /
                starting_capital
                -
                1.0
            ),

        "max_gross_exposure":
            float(
                max_gross_exposure
            ),

        "cap_clipped_count":
            int(
                cap_clipped
            ),

        "daily_stopped_count":
            int(
                daily_blocks
            ),

        "dd_reduced_count":
            int(
                dd_reductions
            ),

        "zero_entry_count":
            int(
                zero_entries
            ),

    }


    return (
        trade_df,
        equity_df,
        summary
    )


# ============================================================
# 9. DAILY EQUITY
# ============================================================

def make_daily_returns(
    equity_df,
    starting_capital=1.0
):

    if equity_df.empty:

        return pd.DataFrame(
            columns=[
                "equity",
                "daily_return",
            ]
        )


    eq = (

        equity_df[
            "equity"
        ]

        .groupby(
            equity_df.index.normalize()
        )

        .last()

        .sort_index()

    )


    previous = eq.shift(1)


    previous.iloc[0] = (
        starting_capital
    )


    daily_return = (

        eq
        /
        previous
        -
        1.0

    )


    return pd.DataFrame({

        "equity":
            eq,

        "daily_return":
            daily_return.fillna(
                0.0
            ),

    })


# ============================================================
# 10. PERFORMANCE
# ============================================================

def performance_stats(
    trade_df,
    equity_df,
    starting_capital=1.0
):

    if trade_df.empty:

        return {}


    active = trade_df.loc[

        trade_df[
            "effective_size"
        ]
        >
        EPS

    ].copy()


    if len(active) == 0:

        return {}


    r = active[
        "trade_return"
    ].astype(float)


    gross_profit = float(

        r[
            r > 0
        ].sum()

    )


    gross_loss = float(

        -r[
            r < 0
        ].sum()

    )


    if gross_loss > 0:

        pf = (
            gross_profit
            /
            gross_loss
        )

    elif gross_profit > 0:

        pf = np.inf

    else:

        pf = np.nan


    # --------------------------------------------------------
    # Event-level Equity DD
    # --------------------------------------------------------

    if equity_df.empty:

        growth = 0.0

        max_dd = 0.0

    else:

        eq = equity_df[
            "equity"
        ].astype(float)


        peak = eq.cummax()


        dd = (

            eq
            /
            peak
            -
            1.0

        )


        max_dd = float(
            dd.min()
        )


        ending_equity = float(
            eq.iloc[-1]
        )


        growth = (

            ending_equity
            /
            starting_capital
            -
            1.0

        )


    # --------------------------------------------------------
    # Daily stats
    # --------------------------------------------------------

    daily = make_daily_returns(

        equity_df,

        starting_capital

    )


    daily_return = daily[
        "daily_return"
    ]


    if len(daily_return) > 0:

        worst_day = float(
            daily_return.min()
        )


        q05 = float(
            daily_return.quantile(
                0.05
            )
        )


        tail = daily_return.loc[

            daily_return <= q05

        ]


        cvar95 = float(

            -tail.mean()

        ) if len(tail) else np.nan


        std = float(
            daily_return.std(
                ddof=1
            )
        )


        if std > 0:

            sharpe = (

                float(
                    daily_return.mean()
                )

                /
                std

                *

                np.sqrt(
                    252
                )

            )

        else:

            sharpe = np.nan

    else:

        worst_day = np.nan

        cvar95 = np.nan

        sharpe = np.nan


    if max_dd < 0:

        return_to_dd = (

            growth

            /
            abs(
                max_dd
            )

        )

    else:

        return_to_dd = np.nan


    return {

        "trades":
            int(
                len(trade_df)
            ),

        "active_trades":
            int(
                len(active)
            ),

        "mean_size":
            float(
                active[
                    "effective_size"
                ].mean()
            ),

        "win_rate":
            float(
                (
                    r > 0
                ).mean()
            ),

        "avg_return":
            float(
                r.mean()
            ),

        "profit_factor":
            float(
                pf
            ),

        "growth":
            float(
                growth
            ),

        "max_dd":
            float(
                max_dd
            ),

        "return_to_dd":
            float(
                return_to_dd
            ),

        "daily_sharpe":
            float(
                sharpe
            ),

        "worst_day":
            float(
                worst_day
            ),

        "daily_cvar_95":
            float(
                cvar95
            ),

    }


# ============================================================
# 11. POLICY SCORE
# ============================================================

def risk_policy_score(
    stats
):

    if not stats:

        return -np.inf


    pf = stats.get(
        "profit_factor",
        np.nan
    )


    growth = stats.get(
        "growth",
        np.nan
    )


    dd = abs(
        stats.get(
            "max_dd",
            np.nan
        )
    )


    cvar = stats.get(
        "daily_cvar_95",
        np.nan
    )


    if not all(

        np.isfinite(x)

        for x in [

            pf,
            growth,
            dd,
            cvar,

        ]

    ):

        return -np.inf


    # Edgeが消えるPolicyは除外
    if pf <= 1.0:

        return -np.inf


    # --------------------------------------------------------
    # 固定されたMeta Score
    #
    # 高Growth
    # 小DD
    # 小Tail Risk
    #
    # を評価
    # --------------------------------------------------------

    score = (

        growth

        -

        0.75
        *
        dd

        -

        0.50
        *
        cvar

    )


    return float(
        score
    )


# ============================================================
# 12. SIMULATOR SANITY TEST
# ============================================================

print()
print("=" * 100)
print("SIMULATOR SANITY TEST")
print("=" * 100)


no_risk_map = {

    int(year):
        "NO_RISK"

    for year in years

}


sanity_trades, sanity_equity, sanity_summary = simulate_policy(

    trades,

    no_risk_map,

    INITIAL_CAPITAL

)


sanity_stats = performance_stats(

    sanity_trades,

    sanity_equity,

    INITIAL_CAPITAL

)


print(
    "Simulation completed successfully."
)

print(
    "Trade records:",
    len(sanity_trades)
)

print(
    "Ending Capital:",
    sanity_summary[
        "ending_capital"
    ]
)

print(
    "Growth:",
    sanity_stats[
        "growth"
    ]
    *
    100,
    "%"
)

print(
    "PF:",
    sanity_stats[
        "profit_factor"
    ]
)

print(
    "Max DD:",
    sanity_stats[
        "max_dd"
    ]
    *
    100,
    "%"
)

print(
    "Maximum Gross Exposure:",
    sanity_summary[
        "max_gross_exposure"
    ]
)


# ============================================================
# 13. STATIC RISK POLICY COMPARISON
#
# 診断用のみ。
#
# ここから最終Policyを選ばない。
# ============================================================

print()
print("=" * 100)
print("STATIC POLICY COMPARISON")
print("(DIAGNOSTIC ONLY)")
print("=" * 100)


static_rows = []


for policy_name in RISK_POLICIES:


    policy_map = {

        int(year):
            policy_name

        for year in years

    }


    rec, eq, sim = simulate_policy(

        trades,

        policy_map,

        INITIAL_CAPITAL

    )


    stats = performance_stats(

        rec,

        eq,

        INITIAL_CAPITAL

    )


    static_rows.append({

        "policy":
            policy_name,

        "score":
            risk_policy_score(
                stats
            ),

        **stats,

        **sim,

    })


static_results = pd.DataFrame(
    static_rows
)


print(

    static_results[
        [

            "policy",

            "growth",

            "profit_factor",

            "max_dd",

            "return_to_dd",

            "daily_sharpe",

            "worst_day",

            "daily_cvar_95",

            "max_gross_exposure",

            "cap_clipped_count",

            "daily_stopped_count",

            "dd_reduced_count",

            "score",

        ]
    ]

    .sort_values(
        "score",
        ascending=False
    )

    .to_string(
        index=False
    )

)


# ============================================================
# 14. META WALK-FORWARD
# ============================================================

warmup_year = years[0]

evaluation_years = years[1:]


print()
print("=" * 100)
print("EXPANDING META WALK-FORWARD")
print("=" * 100)

print(
    "Warm-up year:",
    warmup_year
)

print(
    "Final evaluated years:",
    evaluation_years
)


selected_policy_by_year = {

    int(warmup_year):
        "NO_RISK"

}


selection_rows = []


for test_year in evaluation_years:


    history = trades.loc[

        trades[
            "test_year"
        ]
        <
        test_year

    ].copy()


    candidate_rows = []


    for policy_name in RISK_POLICIES:


        policy_map = {

            int(y):
                policy_name

            for y in sorted(
                history[
                    "test_year"
                ].unique()
            )

        }


        rec, eq, sim = simulate_policy(

            history,

            policy_map,

            INITIAL_CAPITAL

        )


        stats = performance_stats(

            rec,

            eq,

            INITIAL_CAPITAL

        )


        score = risk_policy_score(
            stats
        )


        candidate_rows.append({

            "test_year":
                int(
                    test_year
                ),

            "history_start":
                int(
                    history[
                        "test_year"
                    ].min()
                ),

            "history_end":
                int(
                    history[
                        "test_year"
                    ].max()
                ),

            "policy":
                policy_name,

            "score":
                score,

            "growth":
                stats.get(
                    "growth",
                    np.nan
                ),

            "pf":
                stats.get(
                    "profit_factor",
                    np.nan
                ),

            "max_dd":
                stats.get(
                    "max_dd",
                    np.nan
                ),

            "return_to_dd":
                stats.get(
                    "return_to_dd",
                    np.nan
                ),

            "cvar":
                stats.get(
                    "daily_cvar_95",
                    np.nan
                ),

        })


    candidate_df = pd.DataFrame(
        candidate_rows
    )


    candidate_df = candidate_df.sort_values(

        [
            "score",
            "pf",
        ],

        ascending=[
            False,
            False,
        ]

    )


    winner = candidate_df.iloc[0]


    chosen = str(
        winner[
            "policy"
        ]
    )


    selected_policy_by_year[
        int(
            test_year
        )
    ] = chosen


    selection_rows.extend(
        candidate_rows
    )


    print()
    print("-" * 80)

    print(
        "TEST YEAR:",
        test_year
    )

    print(
        "History:",
        int(
            history[
                "test_year"
            ].min()
        ),
        "->",
        int(
            history[
                "test_year"
            ].max()
        )
    )

    print(
        "Selected:",
        chosen
    )

    print(

        candidate_df[
            [

                "policy",

                "score",

                "growth",

                "pf",

                "max_dd",

                "return_to_dd",

                "cvar",

            ]
        ]

        .to_string(
            index=False
        )

    )


selection_results = pd.DataFrame(
    selection_rows
)


# ============================================================
# 15. FINAL META-OOS
#
# 2020 = Risk Policy選択用Warm-up
# 2021～2026 = 真のMeta OOS評価
# ============================================================

meta_data = trades.loc[

    trades[
        "test_year"
    ].isin(
        evaluation_years
    )

].copy()


baseline_mapping = {

    int(y):
        "NO_RISK"

    for y in evaluation_years

}


risk_mapping = {

    int(y):
        selected_policy_by_year[
            int(y)
        ]

    for y in evaluation_years

}


# ------------------------------------------------------------
# Baseline
# ------------------------------------------------------------

base_rec, base_eq, base_sim = simulate_policy(

    meta_data,

    baseline_mapping,

    INITIAL_CAPITAL

)


base_stats = performance_stats(

    base_rec,

    base_eq,

    INITIAL_CAPITAL

)


# ------------------------------------------------------------
# Risk Engine
# ------------------------------------------------------------

risk_rec, risk_eq, risk_sim = simulate_policy(

    meta_data,

    risk_mapping,

    INITIAL_CAPITAL

)


risk_stats = performance_stats(

    risk_rec,

    risk_eq,

    INITIAL_CAPITAL

)


print()
print("=" * 100)
print("FINAL META-OOS RESULT")
print("=" * 100)


overall = pd.DataFrame(

    {

        "BASELINE": {

            **base_stats,

            **base_sim,

        },

        "RISK_ENGINE": {

            **risk_stats,

            **risk_sim,

        },

    }

).T


print(
    overall.to_string()
)


# ============================================================
# 16. YEARLY FINAL OOS
# ============================================================

print()
print("=" * 100)
print("YEARLY META-OOS RESULTS")
print("=" * 100)


annual_rows = []


for year in evaluation_years:


    ydf = trades.loc[

        trades[
            "test_year"
        ]
        ==
        year

    ].copy()


    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    b_rec, b_eq, b_sim = simulate_policy(

        ydf,

        {
            int(year):
                "NO_RISK"
        },

        INITIAL_CAPITAL

    )


    b = performance_stats(

        b_rec,

        b_eq,

        INITIAL_CAPITAL

    )


    # --------------------------------------------------------
    # Risk
    # --------------------------------------------------------

    chosen_policy = (
        selected_policy_by_year[
            int(year)
        ]
    )


    r_rec, r_eq, r_sim = simulate_policy(

        ydf,

        {
            int(year):
                chosen_policy
        },

        INITIAL_CAPITAL

    )


    r = performance_stats(

        r_rec,

        r_eq,

        INITIAL_CAPITAL

    )


    annual_rows.append({

        "test_year":
            int(year),

        "policy":
            chosen_policy,

        "base_growth":
            b["growth"],

        "risk_growth":
            r["growth"],

        "base_pf":
            b["profit_factor"],

        "risk_pf":
            r["profit_factor"],

        "base_dd":
            b["max_dd"],

        "risk_dd":
            r["max_dd"],

        "base_return_dd":
            b["return_to_dd"],

        "risk_return_dd":
            r["return_to_dd"],

        "base_cvar":
            b["daily_cvar_95"],

        "risk_cvar":
            r["daily_cvar_95"],

        "cap_clipped":
            r_sim[
                "cap_clipped_count"
            ],

        "daily_stopped":
            r_sim[
                "daily_stopped_count"
            ],

        "dd_reduced":
            r_sim[
                "dd_reduced_count"
            ],

    })


annual_results = pd.DataFrame(
    annual_rows
)


print(
    annual_results.to_string(
        index=False
    )
)


# ============================================================
# 17. POLICY FREQUENCY
# ============================================================

print()
print("=" * 100)
print("RISK POLICY SELECTION FREQUENCY")
print("=" * 100)


policy_frequency = (

    annual_results[
        "policy"
    ]

    .value_counts()

)


print(
    policy_frequency
)


# ============================================================
# 18. DAILY RETURNS FOR BOOTSTRAP
# ============================================================

base_daily = make_daily_returns(

    base_eq,

    INITIAL_CAPITAL

)


risk_daily = make_daily_returns(

    risk_eq,

    INITIAL_CAPITAL

)


daily_comparison = pd.concat(

    [

        base_daily[
            "daily_return"
        ].rename(
            "baseline"
        ),

        risk_daily[
            "daily_return"
        ].rename(
            "risk"
        ),

    ],

    axis=1

).fillna(
    0.0
)


daily_comparison[
    "delta"
] = (

    daily_comparison[
        "risk"
    ]

    -

    daily_comparison[
        "baseline"
    ]

)


# ============================================================
# 19. MOVING BLOCK BOOTSTRAP
# ============================================================

def moving_block_bootstrap(
    values,
    block_size=20,
    iterations=5000,
    seed=42
):

    x = np.asarray(
        values,
        dtype=float
    )


    x = x[
        np.isfinite(
            x
        )
    ]


    n = len(x)


    if n == 0:

        return {

            "observed":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "prob_positive":
                np.nan,

        }


    # データが20日未満ならblockを縮める
    block_size = min(
        block_size,
        n
    )


    rng = np.random.default_rng(
        seed
    )


    means = np.empty(
        iterations
    )


    blocks_needed = int(

        np.ceil(
            n
            /
            block_size
        )

    )


    max_start = max(
        0,
        n
        -
        block_size
    )


    for i in range(
        iterations
    ):


        parts = []


        for _ in range(
            blocks_needed
        ):


            if max_start == 0:

                start = 0

            else:

                start = int(

                    rng.integers(

                        0,

                        max_start
                        +
                        1

                    )

                )


            parts.append(

                x[
                    start
                    :
                    start
                    +
                    block_size
                ]

            )


        sample = np.concatenate(
            parts
        )[:n]


        means[i] = np.mean(
            sample
        )


    return {

        "observed":
            float(
                np.mean(x)
            ),

        "ci_low":
            float(
                np.percentile(
                    means,
                    2.5
                )
            ),

        "ci_high":
            float(
                np.percentile(
                    means,
                    97.5
                )
            ),

        "prob_positive":
            float(
                np.mean(
                    means > 0
                )
            ),

    }


bootstrap = moving_block_bootstrap(

    daily_comparison[
        "delta"
    ],

    BOOTSTRAP_BLOCK_DAYS,

    BOOTSTRAP_ITERATIONS,

    RANDOM_SEED

)


print()
print("=" * 100)
print("DAILY RETURN BLOCK BOOTSTRAP")
print("=" * 100)

print(
    "Observed daily delta:",
    bootstrap[
        "observed"
    ]
    *
    100,
    "%"
)

print(
    "95% CI:",
    bootstrap[
        "ci_low"
    ]
    *
    100,
    "%",
    "~",
    bootstrap[
        "ci_high"
    ]
    *
    100,
    "%"
)

print(
    "P(delta > 0):",
    bootstrap[
        "prob_positive"
    ]
    *
    100,
    "%"
)


# ============================================================
# 20. RISK DIAGNOSTICS
# ============================================================

base_growth = (
    base_stats[
        "growth"
    ]
)

risk_growth = (
    risk_stats[
        "growth"
    ]
)


base_dd = abs(
    base_stats[
        "max_dd"
    ]
)

risk_dd = abs(
    risk_stats[
        "max_dd"
    ]
)


base_cvar = (
    base_stats[
        "daily_cvar_95"
    ]
)

risk_cvar = (
    risk_stats[
        "daily_cvar_95"
    ]
)


base_rdd = (
    base_stats[
        "return_to_dd"
    ]
)

risk_rdd = (
    risk_stats[
        "return_to_dd"
    ]
)


# ------------------------------------------------------------
# Growth retention
# ------------------------------------------------------------

growth_retention = (

    risk_growth
    /
    base_growth

    if base_growth > 0

    else np.nan

)


# ------------------------------------------------------------
# Drawdown improvement
# ------------------------------------------------------------

dd_improvement = (

    (
        base_dd
        -
        risk_dd
    )
    /
    base_dd

    if base_dd > 0

    else np.nan

)


# ------------------------------------------------------------
# CVaR improvement
# ------------------------------------------------------------

cvar_improvement = (

    (
        base_cvar
        -
        risk_cvar
    )
    /
    base_cvar

    if (
        np.isfinite(
            base_cvar
        )
        and
        base_cvar > 0
    )

    else np.nan

)


# ------------------------------------------------------------
# Return/DD
# ------------------------------------------------------------

return_dd_improvement = (

    risk_rdd
    /
    base_rdd
    -
    1.0

    if (
        np.isfinite(
            base_rdd
        )
        and
        base_rdd > 0
    )

    else np.nan

)


print()
print("=" * 100)
print("AUTOMATIC RISK DIAGNOSTICS")
print("=" * 100)

print(
    "Baseline Growth:",
    base_growth
    *
    100,
    "%"
)

print(
    "Risk Growth:",
    risk_growth
    *
    100,
    "%"
)

print(
    "Growth Retention:",
    growth_retention
    *
    100,
    "%"
)

print()

print(
    "Baseline Max DD:",
    -base_dd
    *
    100,
    "%"
)

print(
    "Risk Max DD:",
    -risk_dd
    *
    100,
    "%"
)

print(
    "DD Improvement:",
    dd_improvement
    *
    100,
    "%"
)

print()

print(
    "Baseline CVaR95:",
    base_cvar
    *
    100,
    "%"
)

print(
    "Risk CVaR95:",
    risk_cvar
    *
    100,
    "%"
)

print(
    "CVaR Improvement:",
    cvar_improvement
    *
    100,
    "%"
)

print()

print(
    "Baseline Return/DD:",
    base_rdd
)

print(
    "Risk Return/DD:",
    risk_rdd
)

print(
    "Return/DD Improvement:",
    return_dd_improvement
    *
    100,
    "%"
)


# ============================================================
# 21. YEAR STABILITY
# ============================================================

annual_results[
    "risk_growth_positive"
] = (

    annual_results[
        "risk_growth"
    ]
    >
    0

)


annual_results[
    "dd_better"
] = (

    annual_results[
        "risk_dd"
    ].abs()

    <

    annual_results[
        "base_dd"
    ].abs()

)


annual_results[
    "return_dd_better"
] = (

    annual_results[
        "risk_return_dd"
    ]

    >

    annual_results[
        "base_return_dd"
    ]

)


positive_years = int(

    annual_results[
        "risk_growth_positive"
    ].sum()

)


dd_better_years = int(

    annual_results[
        "dd_better"
    ].sum()

)


rdd_better_years = int(

    annual_results[
        "return_dd_better"
    ].sum()

)


print()
print("=" * 100)
print("RISK YEAR STABILITY")
print("=" * 100)

print(
    "Evaluated years:",
    len(
        annual_results
    )
)

print(
    "Positive Risk Growth:",
    positive_years,
    "/",
    len(
        annual_results
    )
)

print(
    "DD Better:",
    dd_better_years,
    "/",
    len(
        annual_results
    )
)

print(
    "Return/DD Better:",
    rdd_better_years,
    "/",
    len(
        annual_results
    )
)


# ============================================================
# 22. FINAL DECISION
# ============================================================

print()
print("=" * 100)
print("FINAL DECISION")
print("=" * 100)


clear_value = (

    np.isfinite(
        growth_retention
    )

    and

    np.isfinite(
        dd_improvement
    )

    and

    np.isfinite(
        return_dd_improvement
    )

    and

    growth_retention >= 0.90

    and

    (
        dd_improvement >= 0.10

        or

        (
            np.isfinite(
                cvar_improvement
            )

            and

            cvar_improvement >= 0.10
        )
    )

    and

    return_dd_improvement >= 0.10

)


risk_reduction_value = (

    np.isfinite(
        growth_retention
    )

    and

    growth_retention >= 0.80

    and

    (
        (
            np.isfinite(
                dd_improvement
            )

            and

            dd_improvement >= 0.10
        )

        or

        (
            np.isfinite(
                cvar_improvement
            )

            and

            cvar_improvement >= 0.10
        )
    )

)


if clear_value:

    decision = (
        "RISK ENGINE HAS CLEAR META-OOS VALUE"
    )

    next_step = (
        "Risk EngineをChampion v1へ組み込み、"
        "次はMachine Learning Model Tournamentへ進む。"
    )


elif risk_reduction_value:

    decision = (
        "RISK ENGINE REDUCES RISK, "
        "BUT RETURN COST EXISTS"
    )

    next_step = (
        "Risk Engineは候補として保存するが、"
        "固定採用せずML Model Tournamentへ進む。"
    )


else:

    decision = (
        "NO CLEAR META-OOS RISK ENGINE VALUE"
    )

    next_step = (
        "Risk設定を結果に合わせて再最適化せず、"
        "現在のAdaptive Position Sizingを維持して"
        "ML Model Tournamentへ進む。"
    )


print(
    "RESULT:",
    decision
)

print()

print(
    "NEXT:",
    next_step
)


# ============================================================
# 23. SAVE ALL RESULTS
# ============================================================

output_dir = (

    Path.cwd()

    /

    (
        "risk_engine_v2_"

        +

        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

    )

)


output_dir.mkdir(

    parents=True,

    exist_ok=False

)


trades.to_csv(

    output_dir
    /
    "standardized_source_trades.csv",

    index=False

)


static_results.to_csv(

    output_dir
    /
    "static_policy_comparison.csv",

    index=False

)


selection_results.to_csv(

    output_dir
    /
    "meta_walk_forward_selection.csv",

    index=False

)


annual_results.to_csv(

    output_dir
    /
    "annual_meta_oos_results.csv",

    index=False

)


base_rec.to_csv(

    output_dir
    /
    "baseline_trade_records.csv",

    index=False

)


risk_rec.to_csv(

    output_dir
    /
    "risk_engine_trade_records.csv",

    index=False

)


daily_comparison.to_csv(

    output_dir
    /
    "daily_return_comparison.csv"

)


overall.to_csv(

    output_dir
    /
    "overall_results.csv"

)


summary_text = f"""
USDJPY RISK ENGINE v2
=====================

Source:
{source_name}

Trades:
{len(trades)}

Years:
{years}

Warm-up:
{warmup_year}

Meta-OOS:
{evaluation_years}

BASELINE
--------
Growth:
{base_stats['growth']}

PF:
{base_stats['profit_factor']}

Max DD:
{base_stats['max_dd']}

Return/DD:
{base_stats['return_to_dd']}

CVaR95:
{base_stats['daily_cvar_95']}


RISK ENGINE
-----------
Growth:
{risk_stats['growth']}

PF:
{risk_stats['profit_factor']}

Max DD:
{risk_stats['max_dd']}

Return/DD:
{risk_stats['return_to_dd']}

CVaR95:
{risk_stats['daily_cvar_95']}


DIAGNOSIS
---------
Growth retention:
{growth_retention}

DD improvement:
{dd_improvement}

CVaR improvement:
{cvar_improvement}

Return/DD improvement:
{return_dd_improvement}

Bootstrap P(delta > 0):
{bootstrap['prob_positive']}

Decision:
{decision}

Next:
{next_step}
"""


with open(

    output_dir
    /
    "risk_engine_summary.txt",

    "w",

    encoding="utf-8"

) as f:

    f.write(
        summary_text
    )


# ============================================================
# 24. FINISHED
# ============================================================

print()
print("=" * 100)
print("FINISHED")
print("=" * 100)

print(
    "Saved to:"
)

print(
    output_dir.resolve()
)

print()

print(
    "スクショして送ってほしい箇所:"
)

print(
    "STATIC POLICY COMPARISON"
)

print(
    "EXPANDING META WALK-FORWARD"
)

print(
    "FINAL META-OOS RESULT"
)

print(
    "YEARLY META-OOS RESULTS"
)

print(
    "AUTOMATIC RISK DIAGNOSTICS"
)

print(
    "RISK YEAR STABILITY"
)

print(
    "FINAL DECISION"
)
